In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/S09_logistica_glm"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesión 9 — Regresión Logística + Modelos Lineales Generalizados (GLM)

**Curso:** Herramientas para la Ciencia de Datos, Facultad de Negocios, UPC
**Programa:** Administración y Ciencia de Datos para Negocios

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

> Curso **Herramientas para la Ciencia de Datos**, Facultad de Negocios, Administración y Ciencia de Datos para Negocios — UPC.
> Español impersonal, fechas DD/MM/YYYY, UTF-8. Las cifras provienen de la investigación verificada de la sesión.

> **Cómo se abre este cuaderno.** El curso lo distribuye por **Google Drive**: en la
> carpeta compartida, clic derecho sobre el archivo → *Abrir con* → *Google
> Colaboratory*. Conviene empezar por **Archivo → Guardar una copia en Drive** para
> conservar el trabajo. No se requiere cuenta de GitHub ni instalar nada en el equipo:
> los datos de la sesión viajan dentro del propio cuaderno.
> **Carpeta del curso en Drive (Pregrado):** https://drive.google.com/drive/folders/1-YJxRt0n-UZwQCu03Lls2LGUYz6KMsl2


## 9.1 — ¿Por qué no se puede predecir una variable binaria mediante una recta? El caso de negocio que hila toda la sesión

Una empresa necesita **anticipar un sí o un no antes de que ocurra**: ¿este cliente se fuga?, ¿este paciente desarrolla cardiopatía?, ¿este casco sufrirá un incidente? La regresión lineal no sirve —predice cualquier número real, no una probabilidad en (0, 1)—. La **regresión logística** y, más en general, los **GLM** resuelven exactamente eso, y además entregan un lenguaje para negocio (el **odds ratio**) y una palanca de decisión (el **umbral por costo**). Toda la sesión avanza sobre ese caso: *modelar la probabilidad de un evento, interpretarla y convertirla en una decisión que minimiza el costo del error.*

## 1. Objetivos de aprendizaje
Al terminar la sesión, el estudiante:

- Ajusta modelos de **regresión logística** (binaria, y reconoce la multinomial y la ordinal) e interpreta sus coeficientes en términos de **odds** y **odds ratio** = exp(β).
- Comprende cómo la regresión se extiende a la **familia exponencial** mediante los **GLM** (Poisson para conteos, binomial, gamma) y su **función de enlace**.
- Evalúa clasificadores con la **matriz de confusión**, la curva **ROC** y el **AUC**, y **elige el umbral de decisión según el costo del error**.
- **Diagnostica** un GLM (deviance, pseudo-R², **sobre-dispersión**) y comunica el resultado como una decisión de negocio.

## 2. Mapa de la sesión: nueve capítulos en dos clases

La sesión ocupa **dos clases**. Cada capítulo lleva un código —9.1 a 9.9— que es **el mismo** en el sílabo, en la guía del docente, en la guía de laboratorio y en las diapositivas, de modo que se pueda pasar de un material a otro sin traducir numeraciones.

**JUEVES — 145 min de contenido** (bloque A1 de 75, receso de 15, bloque A2 de 70; antes, 20 min de control sobre la Sesión 8)

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **9.1** | ¿Por qué no se puede predecir una variable binaria mediante una recta? | «El caso de negocio que hila toda la sesión» y «¿por qué no una OLS?» |
| **9.2** | ¿Qué son el logit, los odds y el odds ratio? | «Teoría guiada» — logit, odds, verosimilitud, deviance, ROC y AUC |
| **9.3** | ¿Cómo se estima el modelo cuando no existe solución cerrada? | «Teoría guiada» — logit, odds, verosimilitud, deviance, ROC y AUC |
| **9.4** | ¿Cómo se mide un clasificador? | «Teoría guiada» — logit, odds, verosimilitud, deviance, ROC y AUC |
| **9.5** | ¿Se sostiene con datos reales? La réplica sobre SAheart | «Réplica de ESL Tabla 4.2 sobre SAheart» — laboratorio, pasos 0 a 5 |

**VIERNES — 120 min corridos**

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **9.6** | ¿Cuándo se puede confiar, y qué procede si la variable no es binaria? | «GLM Poisson con offset» y «Supuestos: cómo identificarlos y corregirlos» |
| **9.7** | ¿Cómo se verifica que el resultado es real? | «Verificación desde la base (recomputado ≈ paper ≈ Excel)» |
| **9.8** | ¿Qué decisión habilita? Abandono de clientes en telecomunicaciones | «Modelo de churn en telco» — laboratorio, paso 6 — y «Laboratorio por industria» |
| **9.9** | ¿Qué no se puede afirmar, y qué sigue en S10? | «Drills, entregable y conexión con S10» |

> El **control** de esta sesión se resuelve en aula, en la franja de 20 minutos del jueves siguiente, y cubre **los nueve capítulos**, de los dos días.


<a id="indice"></a>

## Mapa de la sesión — índice del cuaderno

Cada entrada enlaza a su sección. Toda sección numerada abre con un banner **Sección N de 9** y ofrece un enlace **[↑ índice]** para regresar aquí.

**Preliminares**
- [Cómo leer este cuaderno](#como-leer)
- [Preparación del entorno](#preparacion)

**Secciones**
1. [Teoría guiada — del logit a la decisión](#sec-1)
2. [Réplica de ESL Tabla 4.2 — logística de cardiopatía (SAheart)](#sec-2)
3. [GLM Poisson con offset — tasa de incidentes (ships)](#sec-3)
4. [Laboratorio de negocio — churn en telco (Telco)](#sec-4)
5. [Exportación a Excel y figuras de resultados](#sec-5)
6. [Tablero consolidado (leído del Excel)](#sec-6)
7. [Laboratorio por industria — el método en 5 sectores](#sec-7)
8. [Construcción del pipeline de churn desde cero](#sec-8)
9. [Drills, entregable y conexión con S10](#sec-9)

**Supuestos**
- [Supuestos — cómo identificarlos y corregirlos](#supuestos)

**Anexo**
- [Anexos — desarrollo matemático (A1–A5)](#anexos)

<a id="como-leer"></a>

## Cómo leer este cuaderno

**❓ Qué se quiere averiguar** — abre cada resultado importante del cuaderno: la pregunta que ese número responde, qué decisión depende de ella y **qué significaría cada resultado posible, dicho antes de ver la cifra**. Conviene detenerse ahí y responder mentalmente antes de ejecutar: un dato solo informa a quien traía una pregunta.

**Operativo vs benchmark (regla de oro de la sesión).** El **valor OPERATIVO** de cada resultado es el que produce el entorno del curso (statsmodels / scikit-learn sobre el dataset resuelto): es la columna **`obtenido (venv)`**. Toda cifra publicada (la **Tabla 4.2 de ESL**, los coeficientes de `warpbreaks` en R, la banda de AUC de la literatura) aparece **etiquetada como benchmark** en la columna **`esperado`**, nunca mezclada como resultado propio.

**Leyenda de la tabla-contraste.** `Δ = obtenido − esperado`; **✔** = dentro de la tolerancia (o de la banda) declarada; **✗** = fuera. Para el AUC, que es un *handoff* sin «valor verdadero» publicable, el contraste es de **pertenencia a una banda**, no de Δ estricto.

**Convención Excel del curso.** Los resultados se calculan, se **exportan** a `resultados/S09_resultados.xlsx` y **las figuras (y el tablero final) se generan LEYENDO ese Excel**, no desde objetos en memoria. El **tablero consolidado** del cierre re-lee las celdas del Excel para certificar que lo que se muestra es exactamente lo que se guardó.

> Guía del laboratorio: `laboratorio/GUIA_LABORATORIO_S09.docx`. Plantillas: `plantillas/evaluacion_clasificador.docx` y `plantillas/guia_odds_ratio.docx`. Definiciones e interpretación: el glosario de la sesión e la guía de interpretación de resultados.

Convenciones de los bloques que aparecen a lo largo del cuaderno:

- **🔎 Qué hace este código** precede a cada celda de código; **📖 Cómo se lee esta salida** sigue a cada resultado numérico clave.
- **💡** añade la intuición detrás del resultado y **⚠️** marca un supuesto, un riesgo de interpretación o un error frecuente.
- **🖐️ Cálculo manual** rehace la mecánica sin librería y la verifica con `assert`; **🧮 Matemática en el cuerpo** desarrolla la derivación en LaTeX.
- **✅ Verificación desde la base** recomputa el resultado desde los datos crudos y lo cruza con el Excel; **🧱 Construcción desde cero** rearma el procedimiento completo a partir del archivo original.
- **📄 En el paper** indica la procedencia exacta de cada cifra replicada (fuente, sección y página).


<a id="preparacion"></a>

## Preparación del entorno

La primera celda instala las librerías **solo en Google Colab** (versiones fijadas). En ejecución local se salta automáticamente. La segunda configura imports, rutas y los **tres ayudantes** que sostienen el cuaderno: `tabla()` (renderiza toda salida clave), `contraste()` (arma la tabla `obtenido | esperado | Δ | ¿tol?`) y `leer_celda()` (lee un valor del Excel).

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "scikit-learn": "1.6.1",
    "statsmodels": "0.14.6",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


🔎 **Qué hace este código.** La celda anterior instala librerías **solo en Colab**; esta configura el entorno para todo el cuaderno. En prosa, para quien no domina Python:
- `import statsmodels.api as sm` / `smf` — el motor **estadístico** (entrega coeficientes, errores estándar, p-valores, deviance). Se usa en la réplica logística y el GLM Poisson.
- `from sklearn...` — el motor de **machine learning** (`LogisticRegression`, `train_test_split`, `StandardScaler`, `roc_auc_score`, `confusion_matrix`, `roc_curve`). Se usa en el laboratorio de churn.
- `np.random.seed(42)` y `RANDOM_STATE = 42` — **fijan el azar** para que los resultados sean reproducibles.
- Se definen tres ayudantes que reaparecen en toda la sesión: `tabla()` (muestra un resultado como salida, no escondido en un `print`), `contraste()` (arma la tabla `obtenido | esperado | Δ | ¿tol?`) y `leer_celda()` (lee un valor del Excel).

In [ ]:
# Configuración, imports y helpers de contraste
import warnings
warnings.filterwarnings("ignore")   # convergencia/dispersión: avisos esperados, no son errores

from pathlib import Path
import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

import matplotlib
matplotlib.use("Agg")               # backend headless: cada figura se guarda como PNG y se muestra
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paleta del curso (UPC)
UPC_ROJO, UPC_TINTA, UPC_GRIS = "#C8102E", "#1F2A44", "#8A8D8F"
PALETA = [UPC_ROJO, UPC_TINTA, "#E4879C", "#5B6472", "#A31621", "#B0B3B5", "#7A8CA3"]
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})

def mostrar(fig, ruta):
    "Guarda la figura como PNG a 150 dpi y la muestra en el cuaderno (backend Agg)."
    try:
        fig.tight_layout()
    except Exception:
        pass
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close(fig)
    display(Image(str(ruta)))

def tabla(df, titulo=None):
    "Renderiza un DataFrame como SALIDA del cuaderno (nada crítico queda solo en un print)."
    if titulo:
        display(Markdown(f"**{titulo}**"))
    display(df)   # no se devuelve el df para no renderizar la tabla dos veces

def contraste(registros):
    """Arma la tabla-contraste `obtenido | esperado (benchmark) | Δ | ¿dentro de tol.?`.
    Cada registro es un dict:
      - tolerancia simétrica:  {"concepto":.., "obtenido":.., "esperado":.., "tol":.., "fuente":..}
      - banda (p. ej. AUC):    {"concepto":.., "obtenido":.., "banda":(lo,hi),  "fuente":..}
    El valor OPERATIVO (venv) va en 'obtenido'; el BENCHMARK publicado, en 'esperado'/'banda'.
    """
    filas = []
    for r in registros:
        obt = round(float(r["obtenido"]), 4)
        if "banda" in r:
            lo, hi = r["banda"]
            filas.append({
                "Concepto": r["concepto"],
                "Obtenido (venv, operativo)": obt,
                "Esperado (benchmark)": f"banda [{lo}; {hi}]",
                "Δ = obt − esp": "—",
                "¿Dentro de tol.?": "✔" if lo <= obt <= hi else "✗",
                "Fuente del benchmark": r["fuente"],
            })
        else:
            esp, tol = float(r["esperado"]), float(r["tol"])
            d = round(obt - esp, 4)
            filas.append({
                "Concepto": r["concepto"],
                "Obtenido (venv, operativo)": obt,
                "Esperado (benchmark)": f"{esp}  (±{tol})",
                "Δ = obt − esp": d,
                "¿Dentro de tol.?": "✔" if abs(d) <= tol else "✗",
                "Fuente del benchmark": r["fuente"],
            })
    return pd.DataFrame(filas)

def leer_celda(hoja, celda):
    "Lee una celda del Excel de resultados (data_only): el 'obtenido' del tablero sale del Excel."
    from openpyxl import load_workbook
    wb = load_workbook(XLSX, data_only=True)
    return wb[hoja][celda].value

# ---- Localización de carpetas de la sesión (funciona en local, en nbconvert y en Colab) ----
def localizar_sesion():
    aqui = Path.cwd()
    for base in [aqui, *aqui.parents]:
        if base.name == "S09_logistica_glm" and base.name.startswith("S09"):
            return base
        cand = base / "Sesiones" / "S09_logistica_glm"
        if cand.exists():
            return cand
        if base.name.startswith("S09"):
            return base
    return None

SESION = localizar_sesion()
if SESION is None:
    SESION = Path("/content/S09_logistica_glm"); (SESION / "data").mkdir(parents=True, exist_ok=True)
    print("Modo Colab: carpeta de trabajo en", SESION)
else:
    print("Sesión localizada en:", SESION)

DATA = SESION / "data"
RESULTADOS = SESION / "resultados"; RESULTADOS.mkdir(exist_ok=True)
FIGURAS = SESION / "figuras"; FIGURAS.mkdir(exist_ok=True)
XLSX = RESULTADOS / "S09_resultados.xlsx"

# Mirrors públicos verificados 18/07/2026 (respaldo si el archivo local no está, p. ej. en Colab)
URL_SA = ("https://raw.githubusercontent.com/empathy87/"
          "The-Elements-of-Statistical-Learning-Python-Notebooks/master/"
          "data/South%20African%20Heart%20Disease.txt")
URL_SHIPS = "https://vincentarelbundock.github.io/Rdatasets/csv/MASS/ships.csv"
URL_TELCO = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
             "master/data/Telco-Customer-Churn.csv")

def _cargar(nombre_local, url):
    p = DATA / nombre_local
    if p.exists():
        return pd.read_csv(p)
    print("Archivo local no encontrado; descargando de mirror:", url)
    return pd.read_csv(url)

def cargar_saheart():
    "SAheart (ESL cap. 4; Rousseauw et al. 1983). 462 varones, target chd."
    df = _cargar("SAheart.data", URL_SA)
    df.columns = [c.strip() for c in df.columns]
    return df

def cargar_ships():
    "ships (McCullagh & Nelder 1989, Sección 6.3.2): incidentes de daño por olas con exposición service."
    return _cargar("ships.csv", URL_SHIPS)

def cargar_telco():
    "Telco Customer Churn (IBM 2019): 7 043 clientes, target Churn."
    return _cargar("telco_churn.csv", URL_TELCO)

print("Librerías cargadas. statsmodels", sm.__version__,
      "| helpers: tabla(), contraste(), leer_celda().")
print("Resultados ->", RESULTADOS)

<a id="sec-1"></a>

---

## 9.2 a 9.4 — ¿Qué son el logit, los odds y el odds ratio, cómo se estima el modelo y cómo se mide un clasificador? Teoría guiada — del logit a la decisión (Sección 1 del cuaderno)

> 🧭 **Sección 1 de 9 — En esta sección:** se fija el andamiaje conceptual (logit, MLE, odds ratio, GLM y evaluación con ROC/umbral) con minidemostraciones ilustrativas. — [↑ índice](#indice)


### Objetivo
Fijar el andamiaje conceptual que sostiene toda la sesión.

### Pasos del procedimiento
1. De la lineal a la logística: por qué una probabilidad exige (0, 1) y cómo la sigmoide lo garantiza.
2. Estimación por **máxima verosimilitud**; **deviance** y **pseudo-R²** de McFadden.
3. Interpretación en **odds**; logística multinomial/ordinal; **regularización L1/L2**.
4. **GLM**: familia exponencial, función de enlace; **Poisson/binomial/gamma**; **sobre-dispersión**.
5. Evaluación: **matriz de confusión**, **umbral por costo**, **ROC** y **AUC**.

### Resultado esperado
Cada bloque cierra con una **lectura de negocio**. Las minidemostraciones son **ilustrativas** (datos sintéticos), no la réplica. Base documental: el glosario de la sesión e la guía de interpretación de resultados.

### ¿Por qué regresión logística (o Poisson) y no una regresión lineal (OLS)? — capítulo 9.1 (subsección 1.0)

Antes de ajustar nada conviene justificar la herramienta. La pregunta de negocio de esta sesión es predecir un **evento** (¿se fuga?, ¿tiene cardiopatía?) o **contar** algo (incidentes por casco). En ambos casos la **regresión lineal por mínimos cuadrados (OLS**, S03–S05) *no* es la herramienta correcta, y conviene entender por qué.

**Caso binario: logística en vez de OLS.** Si se codifica el objetivo como 0/1 y se ajusta un OLS (el llamado *modelo de probabilidad lineal*), la recta $\hat p = \beta_0 + \beta_1 x$ **viola** varios supuestos a la vez:

| Supuesto que el OLS viola sobre una binaria | Qué pasa en la práctica |
|---|---|
| **Predicciones dentro de [0, 1]** | la recta no está acotada: en valores extremos de $x$ estima "probabilidades" **negativas o mayores que 1**, sin sentido. |
| **Normalidad de los errores** | con $y \in \{0,1\}$ el residuo solo toma dos valores ($1-\hat p$ o $-\hat p$); no es normal, así que los estadísticos $t$/$F$ y sus IC dejan de ser válidos. |
| **Homocedasticidad (varianza constante)** | la varianza de un Bernoulli es $p(1-p)$, que **depende de $x$**; el OLS supone varianza constante, de modo que los errores estándar quedan mal calculados (**heterocedasticidad**). |
| **Linealidad en la probabilidad** | el OLS impone que un mismo $\Delta x$ mueva la probabilidad lo mismo en todo el rango; en la realidad el efecto se **satura** cerca de 0 y de 1 (forma de S). |

La **logística** resuelve los cuatro problemas: modela el **logit** (log-odds) como algo lineal y lo devuelve a probabilidad con la **sigmoide**, que por construcción queda en $(0,1)$; se estima por **máxima verosimilitud** (no por mínimos cuadrados), con lo que asume la varianza correcta del Bernoulli; y captura la saturación en forma de S. La figura siguiente contrasta ambas sobre los mismos datos binarios.

**Caso de conteos: Poisson en vez de OLS.** Para una respuesta de conteo $y \in \{0, 1, 2, \dots\}$ (incidentes, reclamos, visitas) el OLS también falla: puede predecir **conteos negativos** (imposibles), supone **varianza constante** cuando en los conteos la dispersión **crece con la media**, y trata como continuo y simétrico algo discreto y asimétrico. El **GLM Poisson** usa el **enlace log** —que fuerza $\mu = e^{\eta} > 0$, nunca negativo— y una distribución cuya **varianza es igual a la media** ($\operatorname{Var}(Y)=\mu$), que describe los conteos mucho mejor. El desarrollo formal de estos supuestos y su diagnóstico está en los **Anexos** (Anexo A4).

🔎 **Qué hace este código.** Genera datos binarios sintéticos y **superpone dos ajustes** para exhibir el problema:
- `np.polyfit(x, y, 1)` ajusta la **recta OLS** (grado 1), el 'modelo de probabilidad lineal'.
- `sm.add_constant(x)` añade una **columna de unos** para estimar el intercepto β₀; sin ella el modelo pasaría forzado por el origen.
- `sm.Logit(y, X).fit(disp=0)` estima la **logística por máxima verosimilitud**; `disp=0` **silencia** el log de iteraciones del optimizador (no cambia el resultado, solo evita el ruido en pantalla).
- `mlog.predict(...)` devuelve la **probabilidad** ya pasada por la sigmoide.
La zona rosa marca el rango **imposible** (p < 0 o p > 1) al que llega la recta OLS y nunca la sigmoide.

In [ ]:
# Figura ILUSTRATIVA — por qué NO usar OLS sobre una binaria: la recta se sale de [0,1]; la sigmoide no
rng_fig = np.random.default_rng(RANDOM_STATE)
x_fig = np.sort(rng_fig.normal(0, 1.4, 60))
p_true = 1 / (1 + np.exp(-(0.4 + 2.2 * x_fig)))
y_fig = (rng_fig.uniform(size=x_fig.size) < p_true).astype(int)

b1, b0 = np.polyfit(x_fig, y_fig, 1)                       # recta OLS (modelo de probabilidad lineal)
ols_line = b0 + b1 * x_fig
mlog = sm.Logit(y_fig, sm.add_constant(x_fig)).fit(disp=0)
p_log = mlog.predict(sm.add_constant(x_fig))

fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.axhspan(1.0, 1.35, color="#F3C6D0", alpha=0.35)        # zona imposible: p > 1
ax.axhspan(-0.35, 0.0, color="#F3C6D0", alpha=0.35)       # zona imposible: p < 0
ax.scatter(x_fig, y_fig, color=UPC_GRIS, s=28, alpha=0.75, label="Datos observados (y = 0 / 1)")
ax.plot(x_fig, ols_line, color=UPC_TINTA, lw=2, ls="--", label="OLS (recta): se sale de [0, 1]")
ax.plot(x_fig, p_log, color=UPC_ROJO, lw=2.4, label="Logística (sigmoide): queda en (0, 1)")
ax.axhline(0, color=UPC_GRIS, lw=0.8); ax.axhline(1, color=UPC_GRIS, lw=0.8)
ax.set_ylim(-0.35, 1.35)
ax.set_xlabel("Predictor  x"); ax.set_ylabel("Probabilidad estimada de  y = 1")
ax.set_title("OLS predice fuera de [0, 1] (zona rosa = imposible); la logística no")
ax.legend(loc="center right", frameon=False, fontsize=9)
mostrar(fig, FIGURAS / "S09_fig_ols_vs_logistica.png")

### De la regresión lineal a la logística: logit, odds y odds ratio — capítulo 9.2 (subsección 1.1)

La regresión lineal predice cualquier número real; una **probabilidad** debe quedar en (0, 1). La logística modela el **logit** (log-odds) como una combinación lineal y lo devuelve a probabilidad con la **sigmoide**:

$$\operatorname{logit}(p)=\ln\frac{p}{1-p}=\beta_0+\beta_1 x_1+\dots \qquad p=\frac{1}{1+e^{-(\beta_0+\dots)}}$$

- **odds** = `p / (1 − p)` (razón de momios): rango [0, ∞); odds = 1 equivale a p = 0,5.
- **odds ratio (OR)** de un predictor = **exp(β)**: cuánto se **multiplican** los odds al subir el predictor una unidad. `OR > 1` = factor de riesgo, `OR < 1` = protector.

🧮 **Matemática paso a paso (en el cuerpo): de la probabilidad al logit y de vuelta.** El desarrollo esencial, aquí donde se usa el concepto (la versión extendida está en el **Anexo A1**).

**Paso 1 — de probabilidad a odds.** Si $p=P(Y=1)$, los *odds* (razón de momios) son
$$\text{odds}=\frac{p}{1-p}\in[0,\infty).$$
Ejemplo: $p=0{,}8\Rightarrow \text{odds}=0{,}8/0{,}2=4$ (se lee "4 a 1").

**Paso 2 — de odds a logit.** El logaritmo natural estira el rango $[0,\infty)$ a toda la recta real:
$$\text{logit}(p)=\ln\frac{p}{1-p}\in(-\infty,+\infty).$$
Ejemplo: $\ln 4=1{,}386$.

**Paso 3 — el modelo lineal vive en el logit.** Se iguala el logit a un predictor lineal $\eta$:
$$\ln\frac{p}{1-p}=\eta=\beta_0+\beta_1x_1+\dots+\beta_kx_k.$$

**Paso 4 — de vuelta a probabilidad (sigmoide).** Se despeja $p$ invirtiendo el logit:
$$\frac{p}{1-p}=e^{\eta}\;\Rightarrow\;p=\frac{e^{\eta}}{1+e^{\eta}}=\frac{1}{1+e^{-\eta}}=\sigma(\eta).$$
Como $0<\sigma(\eta)<1$ para todo $\eta$, la predicción **siempre** es una probabilidad válida (la recta del OLS no lo garantiza).

**Paso 5 — odds ratio.** Subir $x_j$ en una unidad suma $\beta_j$ al logit, de modo que **multiplica** los odds por $e^{\beta_j}$: ese factor es el **odds ratio** $\text{OR}=e^{\beta_j}$ (derivación en el **Anexo A5**). La demo de esta subsección recorre este ida y vuelta con números.


🔎 **Qué hace este código.** Dibuja la **sigmoide** `p = 1/(1+e^{-η})` sobre un rango de log-odds `η ∈ [-6, 6]` y arma una tablita que hace el **ida y vuelta** entre las tres escalas: probabilidad `p`, `odds = p/(1-p)` y `logit = ln(odds)`. `np.linspace(-6, 6, 300)` crea 300 puntos igualmente espaciados para trazar la curva suave.

In [ ]:
# Demo ILUSTRATIVA: la sigmoide convierte log-odds en probabilidad; y el ida-vuelta odds <-> probabilidad
z = np.linspace(-6, 6, 300)
p = 1 / (1 + np.exp(-z))
fig, ax = plt.subplots(figsize=(5.4, 3.3))
ax.plot(z, p, color=UPC_ROJO, lw=2.2)
ax.axhline(0.5, ls="--", color=UPC_GRIS, lw=1); ax.axvline(0, ls="--", color=UPC_GRIS, lw=1)
ax.set_xlabel("Predictor lineal  η = β₀ + β₁x   (log-odds)")
ax.set_ylabel("Probabilidad estimada  p = σ(η)")
ax.set_title("La sigmoide acota la predicción al intervalo (0, 1)")
mostrar(fig, FIGURAS / "demo_sigmoide.png")

tabla_odds = pd.DataFrame({"p": [0.10, 0.25, 0.50, 0.80, 0.90]})
tabla_odds["odds = p/(1-p)"] = (tabla_odds["p"] / (1 - tabla_odds["p"])).round(3)
tabla_odds["logit = ln(odds)"] = np.log(tabla_odds["p"] / (1 - tabla_odds["p"])).round(3)
tabla(tabla_odds, "De probabilidad a odds y logit (ilustrativo)")

📖 **Cómo leer esta salida.** En la tabla, cada fila es la misma creencia dicha en tres idiomas:
- `p = 0,50` → `odds = 1` (1 a 1) → `logit = 0`. Es el punto de indiferencia (arriba, la sigmoide cruza 0,5 en η = 0).
- `p = 0,80` → `odds = 4` (4 a 1 a favor) → `logit = 1,386`.
- `p = 0,10` → `odds ≈ 0,111` (1 a 9 en contra) → `logit ≈ -2,197`.
Nótese que `odds` va de 0 a ∞ (nunca negativo) y `logit` recorre toda la recta real: por eso el logit es el que se iguala al predictor lineal.

💡 **Intuición.** Los **odds** son el lenguaje de las apuestas: 'odds 3 a 1' significa que el evento es tres veces más probable que su ausencia, es decir `p = 3/4 = 0,75`. Duplicar los odds (de 1 a 2) **no** duplica la probabilidad (de 0,50 pasa a 0,67): la relación es no lineal, y esa es exactamente la curva en S.

**Lectura de negocio.** El logit es el *puente* entre el mundo lineal ya conocido (S03–S05) y la clasificación: se sigue leyendo un modelo lineal, pero sobre la escala de log-odds. Los odds son el lenguaje del riesgo (*"3 a 1 a que este cliente se fuga"*) y el terreno donde los coeficientes se vuelven interpretables como **factores multiplicativos** (OR). Una recta sin acotar predeciría probabilidades < 0 y > 1; la sigmoide lo impide por construcción.

### Máxima verosimilitud, deviance y pseudo-R² (McFadden) — capítulo 9.3 (subsección 1.2)

A diferencia del OLS (mínimos cuadrados), la logística estima sus coeficientes por **máxima verosimilitud (MLE)**: busca los β que hacen más probables los datos observados. De ahí se obtienen:

- **deviance** `D = −2·(ℓ_modelo − ℓ_saturado)`: falta de ajuste (análoga a la suma de cuadrados residual). Se comparan la *null deviance* (solo intercepto) y la *residual deviance*.
- **pseudo-R² de McFadden** `= 1 − ℓ_modelo/ℓ_nulo`: **no** se lee con el criterio del R² de OLS — valores de **0,2–0,4 ya son un ajuste fuerte**.

🔎 **Qué hace este código.** Simula 400 casos binarios cuyo coeficiente verdadero es `1,5`, ajusta un `sm.Logit` por MLE y extrae sus resúmenes de ajuste: `m.params[1]` (el β estimado), `m.llf` (log-verosimilitud del modelo), `m.llnull` (la del modelo solo-intercepto), la **deviance** `-2·llf` y `m.prsquared` (pseudo-R² de McFadden). El objetivo es ver qué entrega la MLE **además** de los coeficientes.

In [ ]:
# Demo ILUSTRATIVA: un Logit mínimo sobre datos sintéticos y sus estadísticas de ajuste
rng = np.random.default_rng(RANDOM_STATE)
x = rng.normal(size=400)
y = (rng.uniform(size=400) < 1 / (1 + np.exp(-(-0.3 + 1.5 * x)))).astype(int)
m = sm.Logit(y, sm.add_constant(x)).fit(disp=0)
ajuste = pd.DataFrame({
    "estadístico": ["coef(x)  (verdadero 1,5)", "log-verosimilitud", "LL nula",
                    "deviance = −2·llf", "pseudo-R² McFadden"],
    "valor": [round(float(m.params[1]), 3), round(float(m.llf), 2), round(float(m.llnull), 2),
              round(float(-2 * m.llf), 2), round(float(m.prsquared), 3)],
})
tabla(ajuste, "Ajuste de un Logit ilustrativo (0,2–0,4 de pseudo-R² ya es fuerte en logística)")

📖 **Cómo leer esta salida.** Fila por fila:
- `coef(x) ≈ 1,5` — la MLE **recupera** el coeficiente con el que se simularon los datos (buena señal de que el método funciona).
- `log-verosimilitud` — valor negativo; cuanto **más cerca de 0**, mejor ajusta el modelo.
- `LL nula` — la log-verosimilitud del modelo con solo intercepto; es el punto de comparación.
- `deviance = -2·llf` — medida de **falta de ajuste** (análoga a la suma de cuadrados residual del OLS): más baja es mejor.
- `pseudo-R² McFadden = 1 − llf/llnull` — mejora sobre el modelo nulo.

💡 **Intuición.** La MLE elige los β que **hacen más probables los datos que realmente se observaron**, como un detective que se queda con la explicación bajo la cual lo ocurrido era lo más esperable. Y el pseudo-R² **no** se lee con el criterio del OLS: un `0,2–0,4` ya indica un ajuste **fuerte** en logística; leer 'solo explica el 25 %' sería un error clásico.

**Lectura de negocio.** La MLE es lo que permite obtener, además de los coeficientes, sus **errores estándar y p-valores** (qué factores importan) y la **deviance** (para comparar modelos). Que sea iterativa explica los avisos de *"no converge"* (separación perfecta, colinealidad): un diagnóstico frecuente en la práctica.

### Interpretar en odds (y una nota sobre multinomial/ordinal y L1/L2) — capítulo 9.2 (subsección 1.3)

Los coeficientes de un `Logit` viven en **log-odds** y **no** se leen directamente: se **exponencian**. Un cambio de `d` unidades multiplica los odds por `exp(d·β)`.

- **Multinomial** (K > 2 categorías sin orden) y **ordinal** (categorías ordenadas) extienden la logística manteniendo la lectura en odds.
- **Regularización L1/L2** (de S05, ahora sobre clasificación): `LogisticRegression` de scikit-learn aplica **L2 por defecto** (`C = 1/λ`). L1 puede llevar coeficientes a 0 (selección). Precaución: los coeficientes regularizados están **contraídos hacia cero** — para el OR "puro" se usa el ajuste sin penalización (`statsmodels.Logit`).

🔎 **Qué hace este código.** Dos operaciones. (1) Convierte un coeficiente `β = 0,045` (por año) a **odds ratio** con `np.exp(β)` y a su versión por **década** `np.exp(10·β)`. (2) Ajusta la misma logística con `LogisticRegression(C=...)` para tres valores de `C` y compara el coeficiente resultante. Clave: en scikit-learn **`C = 1/λ`**, así que **menor `C` = más regularización** (mayor contracción del coeficiente hacia 0).

In [ ]:
# Demo ILUSTRATIVA: de coeficiente (log-odds) a odds ratio, y el encogimiento por regularización L2
beta = 0.045   # p. ej. coeficiente de 'age' (por año)
filas_or = [
    ["exp(β)  por año",   round(float(np.exp(beta)), 4),    f"+{100*(np.exp(beta)-1):.1f}% odds"],
    ["exp(10·β)  década", round(float(np.exp(10*beta)), 4), f"+{100*(np.exp(10*beta)-1):.1f}% odds"],
]
tabla(pd.DataFrame(filas_or, columns=["cambio", "odds ratio", "lectura"]),
      "β → odds ratio (ilustrativo)")

reg = []
for c_reg in [100.0, 1.0, 0.05]:
    clf_demo = LogisticRegression(C=c_reg, max_iter=1000).fit(x.reshape(-1, 1), y)
    reg.append([c_reg, round(float(clf_demo.coef_[0][0]), 3)])
tabla(pd.DataFrame(reg, columns=["C  (menor C = más regularización)", "coef encogido"]),
      "Efecto de la regularización L2 sobre el coeficiente (ilustrativo)")

📖 **Cómo leer esta salida.** Primera tabla: `exp(0,045) = 1,046` significa **+4,6 % en los odds por año**; `exp(10·0,045) = exp(0,45) = 1,568`, es decir **+56,8 % por década** (el efecto se **compone**, no se suma). Segunda tabla: al bajar `C` de 100 a 0,05 el coeficiente **se contrae** hacia 0 — la regularización sacrifica interpretación a cambio de estabilidad predictiva.

💡 **Intuición.** El odds ratio es **multiplicativo**: un OR de 1,046 aplicado 10 veces es `1,046¹⁰ ≈ 1,57`, no `1 + 10·0,046`. Es el mismo mecanismo del interés compuesto. Por eso, para interpretar el efecto 'puro' se usa el modelo **sin penalización** (`statsmodels.Logit`); los coeficientes contraídos de un modelo regularizado subestiman el OR.

**Lectura de negocio.** *"Cada año adicional multiplica por 1,046 los odds"* es más accionable que un coeficiente en log-odds. Un OR de 1,046 **no** es *"+4,6 % de probabilidad"*, sino *"+4,6 % en los odds"* (la sigmoide es no lineal). L1/L2 se reservan para maximizar el poder predictivo; para **interpretar** el efecto se prefiere el modelo sin penalización.

🔎 **Qué hace este código.** Figura de dos paneles con `β₀ = -1` y `β₁ = 0,6`. Panel izquierdo: en la escala **log-odds** el efecto es una **recta** de pendiente β₁. Panel derecho: en la escala **odds** (eje logarítmico, `ax.set_yscale('log')`) el mismo efecto se ve como **escalones iguales**: cada +1 en x multiplica los odds por `exp(β₁)`. Es la traducción visual de 'aditivo en log-odds = multiplicativo en odds'.

In [ ]:
# Figura ILUSTRATIVA — geometría del odds ratio: cada +1 en x MULTIPLICA los odds por exp(β)
beta0, beta1 = -1.0, 0.6                                   # log-odds = β0 + β1·x  ->  OR por unidad = exp(0,6)
xg = np.linspace(0, 6, 200)
odds = np.exp(beta0 + beta1 * xg)                         # odds = exp(log-odds): recta en escala log
OR = float(np.exp(beta1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.4, 3.8))
# Panel izquierdo: en log-odds el efecto es una RECTA (pendiente β1)
ax1.plot(xg, beta0 + beta1 * xg, color=UPC_TINTA, lw=2.2)
ax1.set_xlabel("Predictor  x"); ax1.set_ylabel("logit(p) = ln(odds) = β₀ + β₁x")
ax1.set_title("En log-odds el efecto es lineal (pendiente β₁)")
# Panel derecho: en odds (escala log) el efecto es MULTIPLICATIVO: escalones iguales de factor OR
ax2.plot(xg, odds, color=UPC_ROJO, lw=2.4)
ax2.set_yscale("log")
for xv in [1, 2, 3]:
    o0, o1 = np.exp(beta0 + beta1 * xv), np.exp(beta0 + beta1 * (xv + 1))
    ax2.hlines(o0, xv, xv + 1, color=UPC_GRIS, lw=1.4, ls=":")
    ax2.vlines(xv + 1, o0, o1, color=UPC_GRIS, lw=1.6)
ax2.set_xlabel("Predictor  x"); ax2.set_ylabel("odds = p / (1 − p)   (escala log)")
ax2.set_title(f"Cada +1 en x multiplica los odds por OR = exp(β₁) = {OR:.2f}")
mostrar(fig, FIGURAS / "S09_fig_geometria_odds_ratio.png")

### GLM: familia exponencial, enlace y sobre-dispersión — capítulo 9.6 (subsección 1.4)

Un **GLM** generaliza la regresión lineal con tres piezas: (1) una distribución de la **familia exponencial** para la respuesta, (2) un **predictor lineal** `η = Xβ`, y (3) una **función de enlace** `g(μ) = η`. La regresión lineal, la logística y los modelos de conteo son todos GLM:

| Respuesta | Familia | Enlace canónico | exp(β) se lee como |
|---|---|---|---|
| Continua | Normal (Gaussiana) | identidad | efecto aditivo |
| Binaria / proporción | **Binomial** | **logit** | **odds ratio** |
| Conteo | **Poisson** | **log** | **rate ratio** (cociente de tasas) |
| Positiva y asimétrica | **Gamma** | log | efecto multiplicativo |

- En **Poisson**, con **offset** `log(exposición)` se modela la **tasa** de eventos por unidad de exposición.
- **Sobre-dispersión:** Poisson asume `Var(Y) = μ`. Si la varianza observada es mayor, el diagnóstico rápido es **`deviance / grados de libertad ≫ 1`** (regla práctica φ > 1,5). Ignorarla produce p-valores demasiado optimistas; se corrige con **quasi-Poisson** o **binomial negativa** (se nombra aquí; el ajuste fino es de modelado avanzado).

📄 **En el paper.** El marco que unifica estas familias procede de **Nelder, J. A. & Wedderburn, R. W. M. (1972). «Generalized Linear Models». _Journal of the Royal Statistical Society, Series A_, 135(3):370-384** (DOI 10.2307/2344614). Allí se demuestra que la normal, la binomial, la Poisson y la gamma son casos de una **misma** estructura —familia exponencial + función de enlace + ajuste por IRLS—. En esta sesión, la logística de la Sección 2 **es** el GLM **binomial con enlace logit** y el modelo de conteos de la Sección 3 **es** el GLM **Poisson con enlace log**: por eso el odds ratio (logit) y el rate ratio (Poisson) son la misma lectura multiplicativa `exp(β)` sobre familias distintas.

🔎 **Qué hace este código.** Simula 200 conteos con media `μ = exp(0,4 + 0,6·x)` (por eso son siempre ≥ 0) y ajusta un **GLM Poisson** con `smf.glm('y ~ x', family=sm.families.Poisson())`. La sintaxis `'y ~ x'` es una **fórmula** (estilo R): a la izquierda la respuesta, a la derecha los predictores. Luego reporta el coeficiente, su `exp(β)` (**rate ratio**) y el diagnóstico de sobre-dispersión `deviance / df_resid`.

In [ ]:
# Demo ILUSTRATIVA: un GLM Poisson mínimo y su chequeo de sobre-dispersión
conteos = pd.DataFrame({"x": rng.normal(size=200)})
mu = np.exp(0.4 + 0.6 * conteos["x"])
conteos["y"] = rng.poisson(mu)
pm = smf.glm("y ~ x", data=conteos, family=sm.families.Poisson()).fit()
res_pois = pd.DataFrame({
    "estadístico": ["coef(x)", "rate ratio = exp(β)", "deviance/df"],
    "valor": [round(float(pm.params["x"]), 3), round(float(np.exp(pm.params["x"])), 3),
              round(float(pm.deviance / pm.df_resid), 3)],
    "lectura": ["escala log", "cociente de tasas", "≈ 1 → sin sobre-dispersión (datos Poisson sintéticos)"],
})
tabla(res_pois, "GLM Poisson ilustrativo (deviance/df ≈ 1 cuando el modelo está bien especificado)")

📖 **Cómo leer esta salida.** `coef(x)` está en **escala log**; su `exp(β)` es un **rate ratio** = cuánto se multiplica la **tasa** de eventos por cada +1 en x. `deviance/df ≈ 1` indica un Poisson **bien especificado** (aquí los datos se simularon Poisson, así que debe resultar cercano a 1).

💡 **Intuición.** Un rate ratio de 1,8 se comunica igual que un odds ratio: 'cada punto de x **multiplica por 1,8** la tasa de reclamos'. La única diferencia con la logística es la familia (conteos en vez de sí/no) y el enlace (log en vez de logit).

⚠️ **Supuestos (Poisson).** El supuesto crítico es **media = varianza** (`Var(Y) = μ`). Se **verifica** mirando `deviance/df`: si es ≫ 1 (regla práctica > 1,5) hay **sobre-dispersión** y los errores estándar quedan **subestimados** (p-valores demasiado optimistas). Se **corrige** con quasi-Poisson o binomial negativa. Desarrollo completo en la sección **[Supuestos](#supuestos)**, en la guía de supuestos de la sesión (Parte 2) y en el **Anexo A4**.

**Lectura de negocio.** Un solo marco cubre binarias (logística), **conteos** (Poisson: reclamos, visitas, llamadas), proporciones (binomial) e importes positivos (gamma). El `exp(β)` se comunica igual que el odds ratio: *"cada punto de X multiplica por 1,8 la tasa de reclamos"*. Revisar `deviance/df` **antes** de reportar es un control de calidad básico.

🔎 **Qué hace este código.** Sobre conteos sintéticos superpone, igual que en 1.0 pero para conteos, la **recta OLS** (`np.polyfit`) y el **Poisson con enlace log** (`smf.glm`). Extiende el eje x a valores negativos (`np.linspace(-2.2, 4.0, 200)`) para exhibir que la recta OLS **cruza a conteos negativos** (zona rosa, imposible) mientras que el Poisson se mantiene ≥ 0.

In [ ]:
# Figura ILUSTRATIVA — conteos: el OLS puede predecir negativos y supone varianza constante; el Poisson no
rng_c = np.random.default_rng(7)
x_c = np.linspace(0, 4, 120)
mu_c = np.exp(0.2 + 0.7 * x_c)                             # media verdadera (enlace log): siempre > 0
y_c = rng_c.poisson(mu_c)                                 # conteos: la varianza crece con la media

b1c, b0c = np.polyfit(x_c, y_c, 1)                        # OLS sobre los conteos
xx = np.linspace(-2.2, 4.0, 200)
ols_c = b0c + b1c * xx
pm_c = smf.glm("y ~ x", data=pd.DataFrame({"x": x_c, "y": y_c}),
               family=sm.families.Poisson()).fit()
pois_c = pm_c.predict(pd.DataFrame({"x": xx}))

fig, ax = plt.subplots(figsize=(6.8, 4.0))
ax.axhspan(-6, 0, color="#F3C6D0", alpha=0.35)            # zona imposible: conteo < 0
ax.scatter(x_c, y_c, color=UPC_GRIS, s=20, alpha=0.65, label="Conteos observados")
ax.plot(xx, ols_c, color=UPC_TINTA, lw=2, ls="--", label="OLS: cruza a conteos negativos")
ax.plot(xx, pois_c, color=UPC_ROJO, lw=2.4, label="Poisson (enlace log): siempre ≥ 0")
ax.axhline(0, color=UPC_GRIS, lw=0.8)
ax.set_ylim(-6, float(y_c.max()) + 3)
ax.set_xlabel("Predictor  x"); ax.set_ylabel("Conteo  y")
ax.set_title("Para conteos, el OLS predice negativos; el Poisson (log) no,\ny su varianza crece con la media")
ax.legend(loc="upper left", frameon=False, fontsize=9)
mostrar(fig, FIGURAS / "S09_fig_poisson_vs_ols.png")

### Evaluar el clasificador: matriz de confusión, umbral, ROC y AUC — capítulo 9.4 (subsección 1.5)

Fijado un **umbral** `t` sobre la probabilidad, cada caso se clasifica y se cruza con la clase real en una **matriz de confusión** (VP, VN, FP, FN). De ahí se obtienen precisión, **recall/sensibilidad** y especificidad.

- El **umbral óptimo por costo** es `t* = C_FP / (C_FP + C_FN)`: si un **falso negativo** cuesta mucho más que un falso positivo, `t*` **baja** (se marca a más gente para no perder positivos reales). En el laboratorio, un FN costará **≈ 15×** un FP.
- La **curva ROC** traza sensibilidad vs. FPR barriendo el umbral; el **AUC** la resume en [0,5, 1] (0,5 = azar). El AUC mide el **poder de ordenamiento** del modelo **con independencia del umbral**.

🔎 **Qué hace este código.** Reutiliza las probabilidades del Logit ilustrativo de 1.2 y, para **dos umbrales** (0,5 y 0,3), clasifica y arma la matriz de confusión. La expresión clave es `confusion_matrix(y, (prob >= t)).ravel()`, que devuelve los cuatro conteos en el orden **VN, FP, FN, VP**. `roc_auc_score(y, prob)` calcula el AUC, que **no depende del umbral**.

In [ ]:
# Demo ILUSTRATIVA: mismo modelo, dos umbrales -> distinta matriz de confusión y recall
prob = 1 / (1 + np.exp(-(-0.3 + 1.5 * x)))       # probabilidades del Logit ilustrativo (1.2)
filas_t = []
for t in [0.5, 0.3]:
    tn, fp, fn, tp = confusion_matrix(y, (prob >= t).astype(int)).ravel()
    filas_t.append([t, int(tp), int(fp), int(fn), int(tn), round(tp / (tp + fn), 2)])
tabla(pd.DataFrame(filas_t, columns=["umbral", "VP", "FP", "FN", "VN", "recall"]),
      f"Bajar el umbral sube el recall (AUC ilustrativo = {roc_auc_score(y, prob):.3f}, independiente del umbral)")

📖 **Cómo leer esta salida.** Las dos filas son **el mismo modelo**, solo cambia el umbral. Al bajar de 0,5 a 0,3, `VP` sube y `FN` baja: se **captan más positivos reales** (el recall sube) a cambio de más `FP`. El **AUC no cambia** entre filas: mide el ordenamiento, no el corte.

💡 **Intuición.** El umbral es un **parámetro de sensibilidad** de la decisión, no del modelo. Bajarlo es como bajar el listón de un detector de metales: suena para más gente (más aciertos sobre lo peligroso, pero también más falsas alarmas). El modelo ordena igual; solo se decide dónde cortar.

🔎 **Qué hace este código.** Dibuja directamente (sin datos, solo rectángulos y texto) el **mapa conceptual** de los cuatro resultados de un clasificador —VN, FP, FN, VP— rotulados con su consecuencia de negocio. Es un **esquema**, no un cálculo: fija que el **FN** (no detectar a quien se va) suele ser el error caro.

In [ ]:
# Figura conceptual (ILUSTRATIVA) — mapa de los dos errores de un clasificador (esquema, sin cifras)
fig, ax = plt.subplots(figsize=(6.6, 4.2)); ax.axis("off")
celdas = [((0, 1), "VN\nse queda y\nlo dejo tranquilo", "#DDE3EC"),
          ((1, 1), "FP\nno se iba y lo contacté\n(costo de campaña)", "#F3C6D0"),
          ((0, 0), "FN\nse fuga y NO lo detecté\n(costo alto: cliente perdido)", "#E4879C"),
          ((1, 0), "VP\nse fuga y lo detecté\n(retención a tiempo)", "#DDE3EC")]
for (cx, cy), txt, col in celdas:
    ax.add_patch(plt.Rectangle((cx, cy), 1, 1, facecolor=col, edgecolor="white", lw=3))
    ax.text(cx + 0.5, cy + 0.5, txt, ha="center", va="center", fontsize=9, color=UPC_TINTA)
ax.text(0.5, 2.12, "Predicho: se queda", ha="center", fontsize=10, weight="bold", color=UPC_TINTA)
ax.text(1.5, 2.12, "Predicho: se fuga", ha="center", fontsize=10, weight="bold", color=UPC_TINTA)
ax.text(-0.14, 1.5, "Real:\nse queda", ha="center", va="center", fontsize=10, weight="bold", color=UPC_TINTA, rotation=90)
ax.text(-0.14, 0.5, "Real:\nse fuga", ha="center", va="center", fontsize=10, weight="bold", color=UPC_TINTA, rotation=90)
ax.set_xlim(-0.35, 2); ax.set_ylim(0, 2.3)
ax.set_title("Los dos errores no cuestan igual: el FN es el caro  [esquema conceptual]")
mostrar(fig, FIGURAS / "S09_fig_matriz_confusion_esquema.png")

**Lectura de negocio.** La matriz traduce el clasificador en las **consecuencias** de sus dos errores, que casi nunca cuestan lo mismo. En churn, un **FN** (no detectar a quien se va) suele costar mucho más que un **FP** (contactar a quien se quedaba): por eso se **baja el umbral**. El AUC dice *"qué tan bien ordena"*; la matriz, en un umbral, dice *"qué decisión toma y a qué costo"*.

> **Fuera de alcance (S10).** El tratamiento estadístico del **desbalance de clases** (remuestreo/SMOTE, F1/PR-AUC como foco, LDA/QDA/Naive Bayes) es de la S10; aquí la palanca es el **costo del error** vía el umbral. KNN y SVM son prerequisito (IA I): solo se referencian.

### Mapa de réplica: los 10 targets cuantitativos — capítulo 9.5 (subsección 1.6)

Antes de ejecutar nada, se declara **qué se espera** (según la ficha de la sesión de réplica del paper`). El notebook reproduce **operativamente** los targets logísticos (#1–#4), el AUC de churn (#10) y la deviance de `ships` (#9); `warpbreaks` (#5–#8) se cita como **benchmark etiquetado** de sobre-dispersión fuerte.

| # | Resultado a reproducir | Esperado (benchmark) | Tolerancia | Fuente |
|---|---|:---:|---|---|
| 1 | coef logístico `tobacco` (log-odds) | 0,080 | ±0,01 | ESL Tabla 4.2 |
| 2 | coef logístico `famhist[Present]` | 0,939 | ±0,02 | ESL Tabla 4.2 |
| 3 | coef logístico `age` | 0,043 | ±0,01 | ESL Tabla 4.2 |
| 4 | coef logístico `ldl` | 0,185 | ±0,02 | ESL Tabla 4.2 |
| 1b–4b | odds ratios `exp(β)` | tobacco 1,083, famhist 2,558, age 1,044 (década 1,530), ldl 1,203 | ±0,03 (±0,10 famhist) | derivado |
| 5 | coef Poisson `wool[B]` (warpbreaks) | −0,206 → RR 0,814 | ±0,01 | R `warpbreaks` |
| 6 | coef Poisson `tension[M]/[H]` | −0,321 / −0,518 | ±0,01 | R `warpbreaks` |
| 7 | intercepto Poisson (tasa base) | 3,692 → 40,1 | ±0,02 | R `warpbreaks` |
| 8 | residual deviance `warpbreaks` | 210,4 / 50 gl → φ≈4,2 | ±0,5 | R `warpbreaks` |
| 9 | residual deviance `ships` (offset log-service) | ≈ 38,7 / 25 gl → φ≈1,55 | ±1,0 | McCullagh & Nelder Sección 6.3.2 |
| 10 | **AUC-ROC** del churn (Telco, test) | banda 0,83–0,85 (*handoff*) | banda | cálculo propio (Frontiers 2025 ≈ 0,88) |

<a id="sec-2"></a>

---

## 9.5 — ¿Se sostiene con datos reales? Réplica de ESL Tabla 4.2 — regresión logística de cardiopatía (SAheart) (Sección 2 del cuaderno)

> 🫀 **Sección 2 de 9 — En esta sección:** se ajusta el Logit de `chd` sobre 7 factores de riesgo y se contrastan sus coeficientes con la Tabla 4.2 de ESL (Contraste #1). — [↑ índice](#indice)


### Objetivo
Reproducir el ajuste por MLE de la **cardiopatía coronaria** (`chd`) sobre factores de riesgo en **462 varones** del estudio CORIS (Sudáfrica), tal como aparece en **Hastie, Tibshirani & Friedman (2009), *ESL*, Sección 4.4.2 y Tabla 4.2**, y contrastarlo con el benchmark publicado.

### Pasos del procedimiento
1. Cargar SAheart y verificar dimensiones (462 × 11).
2. Preparar: `famhist` → dummy (`Present = 1`); **7 predictores** de la Tabla 4.2 (`sbp + tobacco + ldl + famhist + obesity + alcohol + age`, **excluye** `adiposity` y `typea`).
3. Ajustar `sm.Logit` por MLE; leer coeficientes, odds ratio, IC 95 % y p-valor.
4. **Contraste #1**: los cuatro coeficientes significativos contra ESL Tabla 4.2 (targets #1–#4).

### Resultado esperado
Con los 7 predictores, el venv reproduce **de forma prácticamente exacta** los coeficientes publicados: `tobacco` 0,080, `famhist` 0,939, `age` 0,043, `ldl` 0,185. El marco que lo sustenta es el de **Nelder & Wedderburn (1972)**: la logística es el **GLM binomial con enlace logit**.

### Qué preguntaban estos trabajos, y por qué usaron lo que usaron — Sección 0 del paper (subsección 2.0)

💡 **Antes de tocar los datos.** Una réplica sin esta pregunta se convierte en mecánica: se ejecutan celdas y se obtiene un número. Lo que sigue explica **qué buscaba cada autor** y **por qué eligió cada pieza de su método**, que es de donde proviene el criterio para elegir un método propio mañana. Esta sesión no descansa sobre un paper único, sino sobre **una pregunta formulada dos veces sobre datos reales y respondida una vez en abstracto**: la formulan el estudio **CORIS** (Rossouw et al., 1983), origen de `SAheart`, y el ejemplo de conteos de **McCullagh & Nelder** (Sección 6.3.2), origen de `ships`; la responde **Nelder & Wedderburn (1972)**. *(Desarrollo completo con las citas verificadas: la ficha de la sesión de réplica del paper, «Sección 0».)*

**La pregunta de CORIS no era «¿quién se infarta?».** El estudio se titula *Coronary risk factor screening in three rural communities*: el objetivo declarado es **tamizar factores de riesgo**, no predecir infartos. La continuación del mismo equipo, diez años después, explicita la decisión que dependía de esa línea de base —establecer «la viabilidad y la eficacia de un programa multifactorial de intervención comunitaria para reducir los niveles de los factores de riesgo de cardiopatía coronaria» (*Int. J. Epidemiol.*, 22(3), 1993)—, y ese programa combinó una campaña de medios con una **intervención interpersonal cara a cara reservada a los individuos de alto riesgo**, que es cara y no alcanza para todos. La pregunta real era **«¿cuánto multiplica el riesgo cada factor, y a quién conviene priorizar con un presupuesto limitado?»**: el mismo problema que la campaña de retención de la Sección 4.

**La pregunta de 1972 no venía con datos.** Nelder y Wedderburn no traían observaciones que explicar: traían un desorden que ordenar. Hasta ese año el caso binario se llamaba *probit analysis*, el de conteos *contingency tables* y el continuo *regresión*, cada uno con su algoritmo y su capítulo aparte. Su resumen enuncia el hallazgo en una frase: «la técnica de regresión lineal ponderada iterativa puede emplearse para obtener estimaciones de máxima verosimilitud de los parámetros cuando las observaciones se distribuyen según alguna familia exponencial y los efectos sistemáticos pueden volverse lineales mediante una transformación adecuada». El artículo cierra con las implicaciones del marco **para el diseño de los cursos de estadística**: la decisión que dependía del resultado era curricular y de software, no clínica.

**Por qué se descarta la recta sobre una respuesta 0/1.** La alternativa era el **modelo lineal de probabilidad** (OLS con `Y` en {0, 1}), tentador por lo interpretable y descartado por dos defectos. De rango: la recta produce probabilidades por debajo de 0 y por encima de 1, y un riesgo de infarto de −0,12 no sirve para priorizar a nadie. De varianza: la de un Bernoulli es `p(1−p)`, máxima en `p = 0,5` y casi nula en los extremos, de modo que **nunca es constante** y los errores estándar, los t y los p-valores del OLS quedan mal calculados. Para CORIS el defecto era fatal por partida doble: no podía ordenar individuos por riesgo ni decidir qué factor merecía la campaña. La celda 10 de la Sección 1.0 muestra ese fallo dibujado.

**Por qué el logit y no el probit.** El estándar para respuestas binarias era el **probit**; la alternativa logística la introdujo Berkson en 1944 y su difusión posterior es el objeto del trabajo de Cramer (2002). Las dos curvas son casi indistinguibles en el rango útil, así que lo que las separa no es el ajuste sino **qué se puede decir de sus coeficientes**: al modelar `log(p/(1−p))` el coeficiente queda en **log-odds** y `exp(β)` es un **odds ratio**, un factor multiplicativo con lectura verbal inmediata. El coeficiente probit no admite esa traducción. **La logística ganó en el mundo aplicado por ser comunicable, no por ajustar mejor**, y por eso toda la sesión gira alrededor del odds ratio.

**Las decisiones restantes, en corto.** Se estima por **máxima verosimilitud** y no por mínimos cuadrados, con IRLS como precio y como puente con el OLS. Se declara **un marco de tres piezas** —familia exponencial, predictor lineal, función de enlace— en vez de cuatro técnicas, y con él llega un diagnóstico común: la deviance. La Tabla 4.2 ajusta **siete** predictores y no los nueve del archivo. En `ships` se modela una **tasa** y no un conteo, con `log(service)` como offset. Y se añade Telco porque ninguno de los dos datasets históricos trae una decisión con costos.

⚠️ **Dos advertencias antes de leer un solo coeficiente.** La primera: `SAheart` es una **muestra retrospectiva con dos controles por cada caso**, así que su ~34,6 % de positivos es un **artefacto del diseño**, no la prevalencia de la población; sus probabilidades predichas ordenan bien a los pacientes, pero no son riesgos poblacionales y por eso el umbral por costo se trabaja sobre Telco y no sobre este dataset. La segunda: la ficha de la sesión de datos advierte que muchos de los varones con cardiopatía **ya habían recibido tratamiento para bajar la presión** y que en algunos casos la medición se tomó **después** de ese tratamiento — de ahí que `sbp` no resulte significativo sin que quepa concluir que la presión no importa. A diferencia de otras réplicas del curso, aquí el número moderno **no debe diferir** del publicado: `statsmodels` resuelve la misma máxima verosimilitud sobre el mismo modelo, y una desviación grande no es diferencia de software sino error de preparación.


🔎 **Qué hace este código.** `cargar_saheart()` lee el dataset (local o de un mirror verificado); `sa['chd'].value_counts()` cuenta cuántos casos hay con y sin cardiopatía; `sa['chd'].mean()` da la **prevalencia** (proporción de positivos). `sa.head(3)` muestra las tres primeras filas para inspeccionar las columnas.

In [ ]:
# Paso 1 — Cargar SAheart y verificar dimensiones
sa = cargar_saheart()
print("SAheart:", sa.shape, "filas x columnas  |  balance chd:",
      dict(sa["chd"].value_counts()), f"->  {sa['chd'].mean()*100:.1f}% positivos")
tabla(sa.head(3), "Primeras filas de SAheart (chd = cardiopatía coronaria; famhist = antecedente familiar)")

📖 **Cómo leer esta salida.** Entran **462 filas × 11 columnas**. El balance de `chd` muestra ~35 % de positivos: no es una clase muy poco frecuente, pero sí desbalanceada (importa el recall, no solo la exactitud). `famhist` aparece como texto (`Present`/`Absent`): habrá que codificarla como 0/1 antes de ajustar.

**Lectura.** Entran **462 filas** y las columnas canónicas. `famhist` es categórica (`Present`/`Absent`) y se codificará como dummy (`Present = 1`). Se **excluyen** `adiposity` y `typea`: la Tabla 4.2 usa solo **7 predictores** (incluir los 9 daría coeficientes distintos y **no** sería la Tabla 4.2). La primera columna `row.names` es un índice, no un predictor (la ficha de la sesión de réplica del paper`).

🧮 **Matemática paso a paso (en el cuerpo): la verosimilitud que este ajuste maximiza.** El `sm.Logit(...).fit()` de más abajo no minimiza cuadrados como el OLS: **maximiza la verosimilitud** (MLE). El desarrollo esencial (versión extendida en el **Anexo A2**).

**Paso 1 — probabilidad de un caso.** Cada paciente $i$ es Bernoulli con $p_i=\sigma(\eta_i)$, $\eta_i=\beta_0+\beta_1\,\text{sbp}_i+\dots$. La probabilidad de su resultado observado $y_i\in\{0,1\}$ es $p_i^{\,y_i}(1-p_i)^{\,1-y_i}$ (vale $p_i$ si $y_i=1$ y $1-p_i$ si $y_i=0$).

**Paso 2 — verosimilitud conjunta.** Como los 462 casos son independientes, se multiplican:
$$L(\beta)=\prod_{i=1}^{n}p_i^{\,y_i}(1-p_i)^{\,1-y_i}.$$

**Paso 3 — log-verosimilitud.** El logaritmo convierte el producto en suma (más estable y sencillo de derivar):
$$\ell(\beta)=\sum_{i=1}^{n}\big[\,y_i\ln p_i+(1-y_i)\ln(1-p_i)\,\big].$$

**Paso 4 — maximizar.** Se buscan los $\hat\beta$ que hacen **máxima** $\ell(\beta)$. Derivando e igualando a cero se llega a las ecuaciones de verosimilitud:
$$\frac{\partial\ell}{\partial\beta}=\sum_{i=1}^{n}(y_i-p_i)\,x_i=X^{\top}(y-p)=\mathbf{0}.$$
No hay fórmula cerrada (cada $p_i$ depende de $\beta$ de forma no lineal): se resuelve **iterando** (Newton–Raphson / IRLS). Por eso a veces "no converge" (separación perfecta, colinealidad).

**Paso 5 — qué más entrega la MLE.** De la curvatura de $\ell$ se obtienen los **errores estándar** y, con ellos, los $z$, los p-valores y los IC de la tabla. El pseudo-R² y la deviance (más abajo) también se calculan a partir de $\ell$.


**❓ Qué se quiere averiguar.** De los siete factores que se le miden a un paciente, ¿cuáles se asocian realmente con la cardiopatía y **cuánto** multiplica cada uno el riesgo?

- **Qué decide:** el resultado es lo que un médico —o un asegurador— puede usar para priorizar: a quién se cita antes a un tamizaje y sobre qué hábito conviene intervenir primero. Un coeficiente en log-odds no le sirve a nadie; un «multiplica por 2,6 los odds» sí.
- **Antes de mirar el resultado:** cada coeficiente se lee exponenciado. Si `exp(β)` resulta **> 1**, el factor aumenta los odds del evento; si resulta **≈ 1**, no aporta nada; si resulta **< 1**, protege. Y hay un segundo filtro que pesa lo mismo en la decisión: si el intervalo de confianza del odds ratio **cruza el 1**, el efecto no se distingue de «sin efecto», por sugerente que sea la cifra puntual. No todos los factores medidos tienen por qué sobrevivir a ese filtro.

🔎 **Qué hace este código.** Es el núcleo de la réplica.
- `sa['famhist'] = (sa['famhist'] == 'Present').astype(int)` — convierte la categórica a **dummy** (Present = 1, Absent = 0).
- `predictores = [...]` — fija los **7 predictores** de la Tabla 4.2 (excluye `adiposity` y `typea` a propósito).
- `sm.add_constant(...)` — añade la columna de unos del intercepto.
- `sm.Logit(y, X).fit(disp=0)` — ajusta por **máxima verosimilitud**; `disp=0` silencia el log de iteraciones.
- `np.exp(logit.params)` y `np.exp(conf_int())` — pasan los coeficientes y sus **intervalos** de log-odds a **odds ratio**.

In [ ]:
# Pasos 2-3 — Preparar (famhist Present=1; 7 predictores) y ajustar el Logit por MLE
sa["famhist"] = (sa["famhist"] == "Present").astype(int)
predictores = ["sbp", "tobacco", "ldl", "famhist", "obesity", "alcohol", "age"]   # excluye adiposity y typea
X_sa = sm.add_constant(sa[predictores]); y_sa = sa["chd"]
logit = sm.Logit(y_sa, X_sa).fit(disp=0)

# Coeficientes (log-odds) -> odds ratio, con IC 95% y p-valor
ci_or = logit.conf_int(); ci_or.columns = ["ic_bajo", "ic_alto"]
tabla_or = pd.DataFrame({
    "coef (log-odds)": logit.params,
    "odds_ratio = exp(β)": np.exp(logit.params),
    "p_valor": logit.pvalues,
    "OR ic_bajo": np.exp(ci_or["ic_bajo"]),
    "OR ic_alto": np.exp(ci_or["ic_alto"]),
}).round(4)

# Valores OPERATIVOS que alimentarán el Excel (targets declarados)
coef_tobacco = float(logit.params["tobacco"])
coef_famhist = float(logit.params["famhist"])
coef_age     = float(logit.params["age"])
coef_ldl     = float(logit.params["ldl"])
print(f"Pseudo-R² McFadden = {logit.prsquared:.4f}   |   deviance (-2·llf) = {-2*logit.llf:.2f}   |   n = {int(logit.nobs)}")
tabla(tabla_or, "Logit de chd — coeficientes, odds ratio, IC 95% y p-valor (SAheart, 7 predictores)")

📖 **Cómo leer esta salida (número por número).** La tabla es el equivalente al `.summary()` de statsmodels, leído por columnas:
- `coef (log-odds)` — el signo da la **dirección**; la magnitud **no** es probabilidad (por eso se exponencia). `age ≈ 0,043`, `famhist ≈ 0,939`.
- `odds_ratio = exp(β)` — la lectura de negocio: `famhist ≈ **2,56×**`, `age ≈ **1,044×**` por año.
- `p_valor` — si es **< 0,05**, el predictor es significativo. Aquí lo son `tobacco`, `ldl`, `famhist` y `age`; **no** `sbp`, `obesity` ni `alcohol`.
- `OR ic_bajo / OR ic_alto` — intervalo de confianza al 95 % del odds ratio; si **cruza 1**, el efecto no es distinguible de 'sin efecto'.
- La línea de `print` da el **pseudo-R² McFadden** (~0,2, fuerte en logística) y la **deviance**.

💡 **Intuición.** `exp(0,043) = 1,044` significa **+4,4 % de odds por año**; acumulado por década es `exp(10·0,0425) = exp(0,425) = 1,53`, es decir **+53 %**. Y `famhist` con OR 2,56 se lee 'tener antecedente familiar **multiplica por 2,6** los odds de cardiopatía frente a no tenerlo'.

⚠️ **Supuestos (regresión logística).** (1) **Independencia** de las observaciones. (2) **Linealidad en el logit**: log-odds lineal en los predictores (no en la probabilidad); se diagnostica con residuos o términos suavizados. (3) **Sin separación perfecta**: si una variable separa perfectamente las clases, la MLE diverge (coeficientes → ±∞) y aparecen avisos de no convergencia. (4) **Sin multicolinealidad severa** (revisar el VIF). **No** se exigen normalidad ni homocedasticidad (eso era del OLS). Detalle y diagnóstico en el **Anexo A4**.

🧮 **Matemática paso a paso (en el cuerpo): deviance y pseudo-R² de McFadden.** El `print` de arriba mostró la `deviance` y el `Pseudo-R² McFadden`; así se construyen a partir de la log-verosimilitud $\ell(\hat\beta)$ del ajuste (versión extendida en el **Anexo A2**).

**Deviance** — falta de ajuste, análoga a la suma de cuadrados residual del OLS:
$$D=-2\big[\ell(\hat\beta)-\ell_{\text{sat}}\big],\qquad\text{en logística binaria }\ell_{\text{sat}}=0\Rightarrow D=-2\,\ell(\hat\beta).$$
Menor $D$ = mejor ajuste. Se comparan la *null deviance* (solo intercepto, con $\ell_0$) y la *residual deviance* (modelo); su diferencia $\approx\chi^2$ contrasta si el bloque de predictores aporta.

**Pseudo-R² de McFadden** — cuánto mejora el modelo sobre "no saber nada" (solo la tasa base):
$$R^2_{\text{McF}}=1-\frac{\ell(\hat\beta)}{\ell_0}.$$
Nunca llega a 1; en logística **0,2–0,4 ya es un ajuste fuerte** (no se lee con el criterio del R² del OLS). La verificación desde la base (Sección 6) recomputa este mismo tipo de número desde los datos.


**Lectura de negocio (odds ratio, drill 1).** Con el modelo operativo:

- **`famhist` (antecedente familiar): OR ≈ 2,56** — tener antecedente **multiplica por ~2,6** los odds de cardiopatía frente a no tenerlo.
- **`age`: OR ≈ 1,044 por año** → **≈ 1,53 por década** (`exp(10·0,0425)`).
- **`tobacco`: OR ≈ 1,083** por kg acumulado; **`ldl`: OR ≈ 1,20** por unidad.
- `sbp`, `obesity` y `alcohol` **no** son significativos (p > 0,05): su IC del OR cruza 1.

Un pseudo-R² de McFadden de ~0,2 **no** es un mal ajuste: en logística eso es fuerte (`INTERPRETACION_RESULTADOS.md Sección 3`).

🖐️ **Cálculo manual — de probabilidad a odds y de coeficiente a odds ratio.** Antes de confiar en `np.exp`, se reconstruye la mecánica y se confirma que coincide:
- **odds desde `p`:** $\text{odds}=p/(1-p)$.
- **odds ratio:** $\text{OR}=e^{\beta}$, que por definición es el **cociente de odds** al subir el predictor una unidad, $\text{odds}(x{+}1)/\text{odds}(x)$ (Anexo A5). Se calcula de dos formas y se verifica contra `np.exp(logit.params)`.


In [ ]:
# 🖐️ A mano: p -> odds, y OR = exp(β) verificado por la DEFINICIÓN (cociente de odds) y contra np.exp
import math

# (1) p -> odds a mano, con un caso concreto
p_demo = 0.80
odds_demo_manual = p_demo / (1 - p_demo)              # 0.8/0.2 = 4.0  ("4 a 1")
assert np.isclose(odds_demo_manual, 4.0)

# (2) OR de 'age' de DOS formas independientes:
b_age = float(logit.params["age"])
or_age_exp = math.exp(b_age)                          # (a) exp(β) directo
#     (b) por DEFINICIÓN: OR = odds(age+1)/odds(age). Subir 'age' 1 unidad suma β_age al logit.
eta_base = 0.3                                         # cualquier log-odds de partida
or_age_def = math.exp(eta_base + b_age) / math.exp(eta_base)   # el caso base se cancela -> = exp(β_age)
or_age_lib = float(np.exp(logit.params["age"]))       # (c) la función de la librería
print(f"OR(age)  exp(β)={or_age_exp:.6f} | definición(cociente de odds)={or_age_def:.6f} | np.exp={or_age_lib:.6f}")
assert np.isclose(or_age_exp, or_age_lib) and np.isclose(or_age_def, or_age_lib)
print("✔ Coinciden: exp(β) ES el factor por el que se multiplican los odds al subir el predictor 1 unidad.")

# (3) mismo cheque para famhist (dummy): OR compara odds(Present) vs odds(Absent)
or_fam_manual = math.exp(float(logit.params["famhist"]))
assert np.isclose(or_fam_manual, float(np.exp(logit.params["famhist"])))
print(f"OR(famhist)={or_fam_manual:.4f}  ->  tener antecedente familiar multiplica ~2,56x los odds vs no tenerlo")


**❓ Qué se quiere averiguar.** ¿Los coeficientes que acaba de producir este entorno son los mismos que ESL publicó en su Tabla 4.2, o se ha replicado un modelo distinto?

- **Qué decide:** de esto depende si el resto del cuaderno se puede leer. Si el ajuste no reproduce el publicado, cualquier odds ratio posterior describe un modelo distinto del que se cree replicar, y la lectura clínica se apoyaría en una cifra propia sin respaldo.
- **Antes de mirar el resultado:** el contraste enfrenta el valor **operativo** del venv contra el **benchmark etiquetado**, con una tolerancia declarada para cada target. Si los cuatro coeficientes caen dentro, la réplica es fiel y las dos cifras conviven sin fundirse. Si alguno se saliera, el sospechoso no sería el paper: sería la preparación —los 7 predictores de la tabla y no los 9 del archivo, `famhist` codificada como `Present = 1`— y habría que corregirla antes de interpretar nada.

🔎 **Qué hace este código.** Llama al ayudante `contraste([...])` con un registro por target: cada uno lleva el valor `obtenido` (el del venv), el `esperado` (benchmark de ESL Tabla 4.2), la `tol` (tolerancia permitida) y la `fuente`. El ayudante calcula `Δ = obtenido − esperado` y marca ✔/✗ según caiga o no dentro de la tolerancia.

In [ ]:
# Contraste #1 — coeficientes OPERATIVOS (venv, 7 predictores) vs ESL Tabla 4.2 (benchmark etiquetado)
contraste_esl = contraste([
    {"concepto": "coef tobacco (log-odds)", "obtenido": coef_tobacco, "esperado": 0.080, "tol": 0.015, "fuente": "ESL Tabla 4.2"},
    {"concepto": "coef famhist[Present]",   "obtenido": coef_famhist, "esperado": 0.939, "tol": 0.03,  "fuente": "ESL Tabla 4.2"},
    {"concepto": "coef age",                "obtenido": coef_age,     "esperado": 0.043, "tol": 0.01,  "fuente": "ESL Tabla 4.2"},
    {"concepto": "coef ldl",                "obtenido": coef_ldl,     "esperado": 0.185, "tol": 0.02,  "fuente": "ESL Tabla 4.2"},
])
tabla(contraste_esl, "Contraste #1 — réplica logística: obtenido (venv) vs esperado (ESL)")

📖 **Cómo leer esta salida.** Columna a columna: `Obtenido (venv)` es el valor propio; `Esperado (benchmark)` trae el número publicado con su ±tolerancia; `Δ` es la diferencia; `¿Dentro de tol.?` resume el veredicto. Los cuatro coeficientes marcan **✔**: la réplica es fiel. Es la regla de oro de la sesión —el valor que se publica es el **operativo**; ESL entra **etiquetado** como referencia, nunca fundido con el propio.

**Lectura: réplica fiel.** Los cuatro coeficientes caen **dentro de tolerancia** (✔): el operativo del venv **coincide** con ESL Tabla 4.2. Esto ilustra la regla de oro de la sesión: el valor que se publica es el **OPERATIVO** (venv); ESL entra **etiquetado como benchmark**, nunca fundido con el propio.

> **Asociación ≠ causa.** Un OR es una asociación **condicional** dentro del modelo, no un efecto causal (la causalidad formal es S12). Se comunica como *factor de riesgo asociado* (`INTERPRETACION_RESULTADOS.md Sección 8`).

📄 **En el paper.** Estos coeficientes replican **Hastie, T., Tibshirani, R. & Friedman, J. (2009), _The Elements of Statistical Learning_, 2.ª ed., Springer — Sección 4.4.2 «Example: South African Heart Disease», Tabla 4.2 (p. 122)**, que publica el ajuste de máxima verosimilitud de `chd` sobre los 7 factores de riesgo. Correspondencia **fila a fila** con la Tabla 4.2 (coeficientes en log-odds):

| Fila del modelo | Publicado en ESL Tabla 4.2 (p. 122) | Lectura |
|---|---:|---|
| Intercept | −4,130 | tasa base |
| sbp | 0,006 | no significativo |
| tobacco | **0,080** | significativo (replicado) |
| ldl | **0,185** | significativo (replicado) |
| famhist[Present] | **0,939** | significativo (replicado) |
| obesity | −0,035 | no significativo |
| alcohol | 0,001 | no significativo |
| age | **0,043** | significativo (replicado) |

La columna `Obtenido (venv)` del Contraste #1 es el valor **operativo**; la columna publicada de la Tabla 4.2 entra **etiquetada como benchmark**. La **matriz de dispersión** de estos mismos factores de riesgo es la **Figura 4.12 (p. 122)** del libro y se reproduce en la Sección 5.2.

<a id="sec-3"></a>

---

## 9.6 — ¿Qué procede si la variable no es binaria? GLM Poisson con offset — tasa de incidentes y sobre-dispersión (ships) (Sección 3 del cuaderno)

> 🚢 **Sección 3 de 9 — En esta sección:** se modela la tasa de incidentes con `offset=log(service)` y se diagnostica la sobre-dispersión (Contraste #2).  —  [↑ índice](#indice)


### Objetivo
Mostrar que la misma familia GLM modela **conteos**: el dataset **ships** (McCullagh & Nelder 1989, Sección 6.3.2) registra **incidentes de daño por olas** en cascos de carga, con **exposición desigual** `service` (meses-buque). Se modela la **tasa** con `offset = log(service)`.

### Pasos del procedimiento
1. Cargar `ships`, filtrar `service > 0` (34 de 40 filas), fijar `type/year/period` como categóricas.
2. Ajustar `GLM Poisson` con enlace log y `offset = log(service)`.
3. Leer coeficientes → **rate ratios** `exp(β)` y la **deviance/gl**.
4. **Contraste #2**: la sobre-dispersión de `ships` contra el benchmark (M&N ≈ 1,55) y contra dos baselines (φ ideal = 1; `warpbreaks` φ ≈ 4,2).

### Resultado esperado
`deviance/gl ≈ 1,55` → **sobre-dispersión leve** (target #9). El `warpbreaks` clásico exhibe una **fuerte** (φ ≈ 4,2): la misma prueba (`deviance/gl`) distingue ambos regímenes.

**❓ Qué se quiere averiguar.** ¿Qué cascos acumulan más incidentes porque sean efectivamente más peligrosos, y no solo porque llevan más meses en el agua?

- **Qué decide:** la respuesta ordena el trabajo de un asegurador o de un astillero: a quién se inspecciona primero y a quién se le tarifica más caro. Una lista ordenada por incidentes brutos penalizaría a los barcos con más servicio por el mero hecho de haber navegado más.
- **Antes de mirar el resultado:** el `offset = log(service)` es la pieza que convierte el conteo en **tasa por mes de servicio**, así que cada `exp(β)` deja de significar «cuántos incidentes más» y pasa a significar «cuántas veces la tasa» del periodo de referencia. Si un rate ratio resulta **> 1**, ese grupo se accidenta más rápido por unidad de exposición y sube en la lista de inspección. Si resulta **≈ 1**, lo que había era exposición, no riesgo — y sin el offset esa distinción resulta invisible.

🔎 **Qué hace este código.** Ajusta el GLM Poisson sobre `ships`.
- `sh[sh['service'] > 0]` — descarta las combinaciones **sin exposición** (34 de 40 filas).
- `astype('category')` en `type/year/period` — las trata como **factores** (genera dummies internas).
- `family=sm.families.Poisson()` — elige la familia de conteos con enlace log.
- **`offset=np.log(sh['service'])`** — la pieza clave: al fijar el log de la exposición como término con coeficiente 1, el modelo pasa de conteos a **tasa por mes de servicio**, y así `exp(β)` es un **cociente de tasas**.
- `deviance / df_resid` — el diagnóstico de sobre-dispersión.

In [ ]:
# Paso 1-3 — GLM Poisson sobre ships con offset log(service): tasa de incidentes por mes de servicio
sh = cargar_ships()
sh = sh[sh["service"] > 0].copy()                     # 34 de 40 filas con exposición > 0
for c in ["type", "year", "period"]:
    sh[c] = sh[c].astype("category")
pois_ships = smf.glm("incidents ~ C(type) + C(year) + C(period)", data=sh,
                     family=sm.families.Poisson(), offset=np.log(sh["service"])).fit()

dev_ships = float(pois_ships.deviance)
gl_ships = int(pois_ships.df_resid)
dev_gl_ships = dev_ships / gl_ships
print(f"n = {len(sh)} combinaciones  |  incidentes totales = {int(sh['incidents'].sum())}  |  "
      f"deviance = {dev_ships:.4f} / gl = {gl_ships}  ->  deviance/gl = {dev_gl_ships:.4f}")

tabla_pois = pd.DataFrame({
    "coef": pois_ships.params,
    "rate_ratio = exp(β)": np.exp(pois_ships.params),
    "p_valor": pois_ships.pvalues,
}).round(4)
tabla(tabla_pois, "GLM Poisson (ships): coeficientes y rate ratios por término (tasa por mes de servicio)")

# --- IC 95% del rate ratio (exp del IC de beta) para citar la incertidumbre del RR — NO escribe en el Excel ---
ic_beta = pois_ships.conf_int()                        # IC 95% de los coeficientes (escala log)
ic_beta.columns = ["inf", "sup"]
tabla_rr_ic = pd.DataFrame({
    "rate_ratio = exp(β)": np.exp(pois_ships.params),
    "IC95%_inf": np.exp(ic_beta["inf"]),
    "IC95%_sup": np.exp(ic_beta["sup"]),
    "p_valor": pois_ships.pvalues,
    "signif (IC excluye 1)": (np.exp(ic_beta["inf"]) > 1) | (np.exp(ic_beta["sup"]) < 1),
}).round(4)
tabla(tabla_rr_ic, "GLM Poisson (ships): rate ratios con IC 95% (exp del IC de beta) y significancia")


📖 **Cómo leer esta salida.** `coef` en escala log; `rate_ratio = exp(β)` es la lectura directa. Un `C(year)[T.70]` con RR ≈ 2,27 significa que los cascos de 1970-74 tuvieron **2,27× la tasa** de incidentes del periodo de referencia. La línea de `print` reporta `deviance ≈ 38,7 / gl = 25 → deviance/gl ≈ 1,55`.

💡 **Intuición.** Sin el `offset`, un casco con más meses de servicio acumularía más incidentes solo por estar más tiempo expuesto; el offset **normaliza por exposición** y permite comparar **tasas**, igual que comparar 'goles por partido' en vez de 'goles totales'.

⚠️ **Supuestos (Poisson con offset).** Media = varianza; **enlace log** correcto; **offset** bien construido (`log` de la exposición); conteos **independientes**. El fallo típico es la **sobre-dispersión**: aquí `deviance/gl ≈ 1,55` la señala **leve** (`warpbreaks`, φ ≈ 4,2, sería severa). La corrección es quasi-Poisson o binomial negativa. Ver la sección **[Supuestos](#supuestos)**, la guía de supuestos de la sesión (Parte 2) y el **Anexo A4**.

**Lectura de negocio (tasa = rate ratio, drill 3).** Con `offset = log(service)`, cada `exp(β)` es un **cociente de tasas por mes de servicio**. Los cascos construidos en **1970-74** tuvieron **≈ 2,3×** la tasa de incidentes frente al periodo de referencia 1960-64 (`C(year)[T.70]` RR ≈ 2,27); 1965-69 ≈ 2,01×. Es la **misma lectura multiplicativa** que el odds ratio, sobre otra familia.

**❓ Qué se quiere averiguar.** El Poisson da por supuesto que la media y la varianza de los conteos coinciden. ¿Se cumple aquí, o los errores estándar que se acaban de leer están mal calibrados?

- **Qué decide:** si el supuesto falla, los p-valores resultan **optimistas** y se declaran significativos factores que no lo son. Quien tarifique con esos p-valores cobrará distinto a grupos que en realidad no se distinguen entre sí.
- **Antes de mirar el resultado:** el diagnóstico cabe en un solo número, `φ̂ = deviance/gl`. Si resulta **≈ 1**, el Poisson está bien especificado y la tabla anterior se lee tal cual. Si resulta **claramente por encima de 1**, hay sobre-dispersión: **leve** en el entorno de 1,5 —basta con advertirlo al leer los p-valores— y **severa** cerca de 4, como en el caso de `warpbreaks`, donde ya toca cambiar de modelo (quasi-Poisson o binomial negativa).

🔎 **Qué hace este código.** Arma dos tablas: la primera contrasta el `deviance/gl` **obtenido** contra el benchmark de McCullagh & Nelder (≈ 1,55); la segunda ubica a `ships` entre dos referencias — el ideal `φ = 1` (Poisson perfecto) y `warpbreaks` con `φ ≈ 4,2` (sobre-dispersión fuerte, benchmark de R **etiquetado**).

In [ ]:
# Contraste #2 — sobre-dispersión de ships: obtenido (venv) vs benchmark (M&N) y vs dos baselines
contraste_disp = contraste([
    {"concepto": "ships deviance/gl (φ)", "obtenido": dev_gl_ships, "esperado": 1.55, "tol": 0.15,
     "fuente": "McCullagh & Nelder Sección 6.3.2"},
])
tabla(contraste_disp, "Contraste #2 — sobre-dispersión de ships vs benchmark")

ref = pd.DataFrame({
    "referencia": ["ships (obtenido, venv)", "Poisson bien especificado (baseline)",
                   "warpbreaks (sobre-dispersión fuerte)"],
    "deviance/gl (φ)": [round(dev_gl_ships, 4), 1.0, 4.2],
    "lectura": ["sobre-dispersión LEVE", "ideal: Var(Y) = μ", "≈ 210,4 / 50 gl (benchmark R)"],
})
tabla(ref, "ships (leve) entre el ideal φ=1 y warpbreaks φ≈4,2 — la misma prueba los distingue")

📖 **Cómo leer esta salida.** La primera tabla marca **✔**: el `φ ≈ 1,55` cae dentro de la tolerancia del benchmark. La segunda ordena los tres regímenes en una escala: `φ = 1` (ideal), `1,55` (ships, leve), `4,2` (warpbreaks, severa). **La misma prueba** (`deviance/gl`) distingue los tres, y esa es la lección: un solo diagnóstico clasifica la calidad del ajuste.

**Lectura: leve vs fuerte.** `ships` con φ ≈ 1,55 es una sobre-dispersión **leve**; `warpbreaks` (φ ≈ 4,2, benchmark R **etiquetado**) es **severa**. En ambos casos los errores estándar del Poisson simple quedan algo **subestimados** (p-valores optimistas); la corrección sería **quasi-Poisson** o **binomial negativa** —se nombra, no se ajusta aquí— (`DEFINICIONES.md Sección 12`, `INTERPRETACION_RESULTADOS.md Sección 4`).

📄 **En el paper.** El dataset **ships** y su deviance de referencia proceden de **McCullagh, P. & Nelder, J. A. (1989), _Generalized Linear Models_, 2.ª ed., Chapman & Hall — Sección 6.3.2 «ships»**: el modelo con `offset = log(service)` tiene una **deviance de referencia ≈ 38,7 sobre 25 gl** (→ `deviance/gl ≈ 1,55`, el valor obtenido arriba). El contrapunto de sobre-dispersión fuerte, **warpbreaks** (φ ≈ 4,2; 210,4/50 gl; coeficientes documentados woolB −0,206, tensionM −0,321, tensionH −0,518), **no proviene de un paper con tabla numerada**: es el conjunto `datasets::warpbreaks` de **base R**, por lo que se etiqueta como **benchmark de R**, no como cita de paper.

<a id="sec-4"></a>

---

## 9.8 — ¿Qué decisión habilita? Laboratorio de negocio: modelo de churn en telco (Telco Customer Churn) (Sección 4 del cuaderno)

> 📉 **Sección 4 de 9 — En esta sección:** se entrena la logística de churn, se evalúa con matriz/ROC/AUC y se elige el umbral por costo del falso negativo (Contrastes #3 y #4).  —  [↑ índice](#indice)


### Objetivo
Cerrar el caso de negocio: con una muestra pública de **IBM** (7 043 clientes, target `Churn`), ajustar una **regresión logística**, evaluarla con **matriz de confusión / ROC / AUC** y **elegir el umbral según el costo del falso negativo**, para recomendar una política de retención.

### Pasos del procedimiento
1. Cargar y limpiar (`TotalCharges` → numérico; `Churn` → 1/0); medir la **prevalencia**.
2. Codificar (one-hot) y particionar train/test estratificado 80/20.
3. Entrenar `LogisticRegression` (L2 por defecto); **AUC en test** (target #10) y matriz a 0,5.
4. **Contraste #3** (AUC vs azar y vs banda) y **#4** (umbral óptimo vs por defecto), con **costo FN/FP = 15×**.
5. Interpretar los OR del churn como **palancas de retención**.

### Resultado esperado
AUC dentro de la **banda 0,83–0,85** (handoff). Como un cliente perdido cuesta **≈ 15×** una llamada de retención, el umbral óptimo cae **muy por debajo de 0,5**: bajarlo a ≈ 0,10 sube el recall de ~0,57 a ~0,95. El **desbalance** (26,6 %) se aborda **por costo del umbral**, no por remuestreo (eso es S10).

**❓ Qué se quiere averiguar.** Antes de entrenar nada: ¿qué exactitud alcanza el clasificador más simple posible, el que asigna a todos los clientes la clase «no abandona»?

- **Qué decide:** ese número es el **piso** contra el que hay que juzgar todo lo que venga después. Sin él, una exactitud del 78 % parece indicar un buen modelo; con él puede descubrirse que el modelo es peor que no hacer nada.
- **Antes de mirar el resultado:** la prevalencia de fuga marca el listón, porque la exactitud del clasificador trivial es exactamente «1 − prevalencia». Si la fuga rondara el 50 %, la exactitud global sería una métrica informativa y ese baseline acertaría la mitad de las veces. Si la fuga resulta **minoritaria**, el mismo baseline acierta en la gran mayoría de los casos **sin detectar ni una sola fuga**: recall 0. En ese escenario la exactitud global queda descartada como criterio, y la sesión se juzga por recall y por costo del error.

🔎 **Qué hace este código.** Prepara Telco.
- `pd.to_numeric(tc['TotalCharges'], errors='coerce')` — fuerza a número; las **11 celdas en blanco** se vuelven `NaN` en vez de romper el proceso (`errors='coerce'`).
- `dropna(subset=['TotalCharges'])` — descarta esas 11 filas; `drop(columns=['customerID'])` quita el identificador (no es predictor).
- `(tc['Churn'] == 'Yes').astype(int)` — codifica el target como 1/0.
- `tc['Churn'].mean()` — la **prevalencia** de fuga.

In [ ]:
# Paso 1 — Cargar y limpiar Telco (TotalCharges: 11 celdas en blanco -> NaN) y medir prevalencia
tc = cargar_telco()
tc["TotalCharges"] = pd.to_numeric(tc["TotalCharges"], errors="coerce")
n_blanco = int(tc["TotalCharges"].isna().sum())
tc = tc.dropna(subset=["TotalCharges"]).drop(columns=["customerID"])
tc["Churn"] = (tc["Churn"] == "Yes").astype(int)
prevalencia_churn = float(tc["Churn"].mean())
tabla(pd.DataFrame({
    "concepto": ["clientes tras limpiar", "celdas en blanco -> NaN", "prevalencia de fuga (churn)"],
    "valor": [tc.shape[0], n_blanco, f"{prevalencia_churn*100:.1f}%"],
}), "Telco listo: dimensiones y prevalencia de churn")

📖 **Cómo leer esta salida.** Quedan ~7 032 clientes tras limpiar; 11 celdas pasaron a `NaN`; la **prevalencia de fuga es ~26,6 %**. Ese número es el **umbral mínimo a superar**: un clasificador que asignara a todos la clase 'no abandona' acertaría ~73,4 % de las veces… detectando **cero** fugas. Por eso la exactitud induce a error y se mirarán recall y costo.

**Lectura: el umbral mínimo a superar.** Con ~26,6 % de fuga, el **baseline trivial** *"no abandona" para todos* logra una exactitud de ~0,734… con **recall 0** (no detecta ninguna fuga). Ese es el contrafactual del modelo: **la exactitud induce a error** cuando la clase positiva es minoritaria; por eso importan recall y costo, no la exactitud global (`INTERPRETACION_RESULTADOS.md Sección 5`).

🔎 **Qué hace este código.** Deja los datos listos para el modelo.
- `pd.get_dummies(..., columns=cat, drop_first=True)` — **one-hot**: convierte cada categórica en columnas 0/1; `drop_first=True` elimina una categoría de referencia para evitar la **colinealidad perfecta** entre dummies (*dummy variable trap*).
- `train_test_split(..., stratify=yt, random_state=42)` — separa 80/20; **`stratify=yt`** conserva la misma prevalencia de churn en train y test; la semilla fija la partición.
- `StandardScaler().fit(Xtr[num])` — estandariza las numéricas **ajustando solo con train** (para no filtrar información del test).

In [ ]:
# Paso 2 — Codificar (one-hot) y particionar train/test estratificado (80/20)
num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = [c for c in tc.columns if c not in num + ["Churn"]]
Xt = pd.get_dummies(tc[num + cat], columns=cat, drop_first=True)   # one-hot; drop_first evita la trampa de la dummy (colinealidad)
yt = tc["Churn"]

Xtr, Xte, ytr, yte = train_test_split(Xt, yt, test_size=0.2, stratify=yt, random_state=RANDOM_STATE)   # stratify=yt conserva la prevalencia en ambos lados
escalador = StandardScaler().fit(Xtr[num])
Xtr = Xtr.copy(); Xte = Xte.copy()
Xtr[num] = escalador.transform(Xtr[num]); Xte[num] = escalador.transform(Xte[num])
print("Matriz de diseño:", Xt.shape[1], "variables (one-hot)  |  train:", Xtr.shape[0], " test:", Xte.shape[0])

💡 **Intuición.** El **one-hot** es como pasar una pregunta de opción múltiple ('¿qué contrato?') a varias de sí/no ('¿es mes a mes?', '¿es a un año?'); se descarta una opción como referencia contra la cual se comparan las demás. Y **estratificar** el split es como repartir dos mazos de cartas manteniendo la misma proporción de figuras en cada mano: así el AUC del test es representativo y reproducible.

**Lectura.** Las numéricas se **escalan** (la logística de sklearn regulariza por defecto: conviene poner las variables en la misma escala) y el split es **estratificado**, de modo que la prevalencia se conserva en train y test. La semilla 42 fija la partición para que el AUC sea reproducible.

**❓ Qué se quiere averiguar.** Con el modelo ya entrenado, ¿cuántas de las fugas reales llega a detectar la política por defecto, la que marca como fuga a quien supere una probabilidad de 0,5?

- **Qué decide:** cada fuga no detectada es un cliente que se marcha sin que nadie lo llame. El recuento de falsos negativos es, literalmente, la lista de clientes que el plan de retención no verá.
- **Antes de mirar el resultado:** conviene distinguir dos conceptos que suelen confundirse. El **AUC** mide **ordenamiento** y no depende del umbral: un valor alto dice que el modelo coloca a los que se fugan por delante de los que se quedan. El **recall al 0,5** mide la **decisión**. Si ambos resultan altos, el corte por defecto sirve. Si el AUC resulta alto pero el recall se queda **en torno a la mitad**, la conclusión no es que el modelo sea malo: es que el corte está mal puesto — ordenar no es decidir.

🔎 **Qué hace este código.** Entrena y evalúa el clasificador de churn.
- `LogisticRegression(max_iter=1000).fit(Xtr, ytr)` — ajusta la logística (con **L2** por defecto, `C = 1,0`); `max_iter` sube el tope de iteraciones para asegurar convergencia.
- `clf.predict_proba(Xte)[:, 1]` — devuelve la **probabilidad de la clase 1 (fuga)**; el `[:, 1]` toma esa segunda columna.
- `roc_auc_score(yte, prob_te)` — el **AUC** en test.
- `confusion_matrix(...).ravel()` — la matriz a umbral 0,5, aplanada en el orden **VN, FP, FN, VP**.

In [ ]:
# Paso 3 — Entrenar la logística (L2 por defecto), AUC en test y matriz de confusión a 0,5
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
prob_te = clf.predict_proba(Xte)[:, 1]           # [:, 1] = P(clase 1 = fuga)
auc_churn = float(roc_auc_score(yte, prob_te))   # AUC = poder de ordenamiento, independiente del umbral
fpr, tpr, _ = roc_curve(yte, prob_te)
cm05 = confusion_matrix(yte, (prob_te >= 0.5).astype(int))
tn, fp, fn, tp = cm05.ravel()                    # ravel(): VN, FP, FN, VP
print(f"AUC-ROC (test) = {auc_churn:.4f}   [OPERATIVO; banda benchmark literatura ≈ 0,84-0,88]")
tabla(pd.DataFrame(
    [["Real: se queda", int(tn), int(fp)], ["Real: se fuga", int(fn), int(tp)]],
    columns=["", "Pred: se queda", "Pred: se fuga"]),
    "Matriz de confusión al umbral por defecto (0,5)")
print(f"recall = {tp/(tp+fn):.3f}  |  precisión = {tp/(tp+fp):.3f}  |  especificidad = {tn/(tn+fp):.3f}")

# --- Incertidumbre del AUC (banda del 0,8361 para que la guía la cite) — NO escribe en el Excel ---
# (a) IC 95% por bootstrap sobre el conjunto de test (remuestreo con reemplazo, semilla fija).
_y_te = yte.to_numpy()
_rng = np.random.default_rng(42)
_auc_boot = np.empty(2000)
for _b in range(2000):
    _i = _rng.integers(0, len(_y_te), len(_y_te))     # índices con reemplazo
    _auc_boot[_b] = roc_auc_score(_y_te[_i], prob_te[_i])
auc_ic95 = np.percentile(_auc_boot, [2.5, 97.5])       # percentiles 2,5 y 97,5
# (b) Rango por validación cruzada estratificada (5 folds), escalando SOLO las numéricas dentro de cada fold (sin fuga).
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
_pipe_cv = Pipeline([("esc", ColumnTransformer([("num", StandardScaler(), num)], remainder="passthrough")),
                     ("logit", LogisticRegression(max_iter=1000))])
auc_cv = cross_val_score(_pipe_cv, Xt, yt, cv=5, scoring="roc_auc")
print(f"AUC (test) = {auc_churn:.4f}  |  IC 95% bootstrap (2000 remuestreos, semilla 42) = "
      f"[{auc_ic95[0]:.4f}; {auc_ic95[1]:.4f}]")
print(f"AUC por validación cruzada (5 folds) = {auc_cv.mean():.4f} ± {auc_cv.std():.4f}  "
      f"(rango {auc_cv.min():.4f}-{auc_cv.max():.4f})")


📖 **Cómo leer esta salida (celda por celda).** La matriz de confusión al 0,5 se lee así:
- **VN** (arriba-izq.): se quedan y se predijo que se quedan (acierto).
- **FP** (arriba-der.): se quedan pero se los marcó como fuga (falsa alarma, costo de campaña).
- **FN** (abajo-izq.): **se fugan y no se detectaron** — el error caro (~159 casos aquí).
- **VP** (abajo-der.): se fugan y se detectaron (retención a tiempo).
Debajo: `recall ≈ 0,58` (solo se capta el 58 % de la fuga), `precisión` y `especificidad`. El `AUC ≈ 0,84` dice que el modelo **ordena bien**, pero ordenar no es decidir: falta elegir el umbral.

⚠️ **Supuestos (clasificación honesta).** El umbral se fija con **validación**, no mirando el test (mover el corte hasta obtener una cifra favorable es **fuga de información**); aquí se reporta en test solo para ilustrar. Se asume además **independencia** de los clientes y linealidad en el logit. El tratamiento del **desbalance** (SMOTE, PR-AUC) es materia de S10; en esta sesión la palanca es el **costo del umbral**. Ver la sección **[Supuestos](#supuestos)**, la guía de supuestos de la sesión (Parte 1) y el **Anexo A4**.

**Lectura.** Con el 0,5 por defecto **no se detectan 159 fugas reales** (FN): el recall es apenas ~0,58. El AUC (~0,84) dice que el modelo **ordena bien**, pero ordenar no es decidir: falta elegir el punto de corte. La matriz, en un umbral, muestra *qué decisión toma y a qué costo* (`INTERPRETACION_RESULTADOS.md Secciones 5 a 7`).

🖐️ **Cálculo manual — la matriz de confusión contando con comparaciones booleanas.** `confusion_matrix` no es una caja negra: cada celda es un **conteo**. Con la verdad `yte` y la predicción al umbral 0,5 se cuentan los cuatro casos y se confirma que igualan `confusion_matrix(...).ravel()` (orden de sklearn: **VN, FP, FN, VP**). De esos cuatro conteos se obtienen, también de forma manual, las métricas del Paso 3:
- $\text{VP}=\#\{\hat y=1,\;y=1\}$, $\;\text{VN}=\#\{\hat y=0,\;y=0\}$, $\;\text{FP}=\#\{\hat y=1,\;y=0\}$, $\;\text{FN}=\#\{\hat y=0,\;y=1\}$.
- $\text{recall}=\dfrac{\text{VP}}{\text{VP}+\text{FN}}$, $\quad\text{precisión}=\dfrac{\text{VP}}{\text{VP}+\text{FP}}$, $\quad\text{especificidad}=\dfrac{\text{VN}}{\text{VN}+\text{FP}}$.


In [ ]:
# 🖐️ A mano: contar VP/VN/FP/FN con comparaciones booleanas y verificar contra confusion_matrix
y_real = yte.to_numpy()                              # verdad (0/1)
y_pred = (prob_te >= 0.5).astype(int)                # decisión al umbral 0,5

VP_m = int(((y_pred == 1) & (y_real == 1)).sum())    # verdaderos positivos
VN_m = int(((y_pred == 0) & (y_real == 0)).sum())    # verdaderos negativos
FP_m = int(((y_pred == 1) & (y_real == 0)).sum())    # falsos positivos
FN_m = int(((y_pred == 0) & (y_real == 1)).sum())    # falsos negativos

tn_lib, fp_lib, fn_lib, tp_lib = confusion_matrix(y_real, y_pred).ravel()   # orden ravel: VN, FP, FN, VP
print(f"a mano   -> VN={VN_m} FP={FP_m} FN={FN_m} VP={VP_m}")
print(f"librería -> VN={tn_lib} FP={fp_lib} FN={fn_lib} VP={tp_lib}")
assert (VN_m, FP_m, FN_m, VP_m) == (int(tn_lib), int(fp_lib), int(fn_lib), int(tp_lib))
print("✔ La matriz 'a mano' coincide con confusion_matrix().")

# Métricas a mano (mismas fórmulas del Paso 3)
recall_m    = VP_m / (VP_m + FN_m)
precision_m = VP_m / (VP_m + FP_m)
especif_m   = VN_m / (VN_m + FP_m)
print(f"recall={recall_m:.3f}  precisión={precision_m:.3f}  especificidad={especif_m:.3f}")
assert np.isclose(recall_m, tp_lib/(tp_lib+fn_lib)) and np.isclose(precision_m, tp_lib/(tp_lib+fp_lib))
print("✔ recall, precisión y especificidad reproducen las del Paso 3.")


📖 **Cómo leer.** Los cuatro números que devuelve `confusion_matrix` son exactamente estas cuatro comparaciones booleanas sumadas; recall, precisión y especificidad son cocientes de esos conteos. Ver la función "por dentro" evita el error clásico de leer la matriz al revés (recordar el orden de `ravel()`: **VN, FP, FN, VP**).


🖐️ **Cálculo manual — el AUC como probabilidad de ranking (Mann–Whitney) y como área bajo la ROC.** `roc_auc_score` resume la curva ROC en un número; su significado exacto es **la probabilidad de que un positivo al azar reciba mayor score que un negativo al azar**:
$$\text{AUC}=\frac{1}{n_{+}\,n_{-}}\sum_{i\in\text{pos}}\sum_{j\in\text{neg}}\Big[\mathbf{1}(s_i>s_j)+\tfrac{1}{2}\,\mathbf{1}(s_i=s_j)\Big].$$
Se calcula de **dos** formas independientes —promedio sobre todos los pares (positivo, negativo), y área bajo la ROC por la regla del trapecio— y se confirma que ambas igualan la función.


In [ ]:
# 🖐️ A mano (1): AUC como probabilidad de ranking (estadístico de Mann–Whitney U)
scores = prob_te
pos = scores[y_real == 1]        # scores de los que SÍ se fugaron
neg = scores[y_real == 0]        # scores de los que NO se fugaron
mayores = (pos[:, None] >  neg[None, :]).sum()       # el positivo puntúa MÁS alto que el negativo
empates = (pos[:, None] == neg[None, :]).sum()       # empate -> cuenta 1/2
auc_ranking = (mayores + 0.5 * empates) / (len(pos) * len(neg))

# 🖐️ A mano (2): AUC como área bajo la ROC por la regla del trapecio (sin np.trapz)
fpr_h, tpr_h, _ = roc_curve(y_real, scores)
auc_trapecio = float(np.sum(np.diff(fpr_h) * (tpr_h[:-1] + tpr_h[1:]) / 2))   # Σ base·(altura media)

auc_lib = roc_auc_score(y_real, scores)              # la función
print(f"AUC  ranking(Mann–Whitney)={auc_ranking:.6f} | trapecios={auc_trapecio:.6f} | roc_auc_score={auc_lib:.6f}")
assert np.isclose(auc_ranking, auc_lib, atol=1e-6) and np.isclose(auc_trapecio, auc_lib, atol=1e-6)
print("✔ Las dos construcciones a mano coinciden con roc_auc_score().")


📖 **Cómo leer.** Un AUC de ~0,84 significa que, al tomar al azar un cliente que se fue y uno que se quedó, el modelo le asigna mayor score al que se fue **~84 % de las veces**. Es *ordenamiento*, no *decisión*: no depende del umbral (por eso el mismo AUC convive con muchas matrices de confusión, una por cada umbral).


🔎 **Qué hace este código.** Contrasta el AUC obtenido contra una **banda** de literatura (0,83–0,85) en vez de un valor puntual: como el AUC depende de la preparación (imputación, split, escalado), no hay un 'valor verdadero' publicable, así que el ✔ se otorga por **pertenencia a la banda**. La segunda tabla lo compara con el azar (0,5).

In [ ]:
# Contraste #3 — AUC del churn: obtenido (venv) vs banda de literatura (handoff) y vs azar
contraste_auc = contraste([
    {"concepto": "AUC-ROC del churn (test)", "obtenido": auc_churn, "banda": (0.83, 0.85),
     "fuente": "banda literatura; handoff (REPLICACION Sección 4 #10)"},
])
tabla(contraste_auc, "Contraste #3 — AUC operativo dentro de la banda de literatura")

tabla(pd.DataFrame({
    "referencia": ["AUC modelo (venv)", "azar (diagonal ROC)"],
    "AUC": [round(auc_churn, 4), 0.5],
    "lectura": [f"supera el azar en +{auc_churn-0.5:.3f}", "piso a batir"],
}), "El modelo ordena muy por encima del azar (el baseline a superar, no un rival serio)")

📖 **Cómo leer esta salida.** La primera tabla marca **✔**: el AUC operativo cae dentro de la banda handoff. La segunda muestra la distancia al azar (`AUC − 0,5`): el modelo ordena **muy por encima** del lanzamiento de una moneda. Advertencia: el azar es el piso a superar, no un rival serio; el rival real es el costo de las decisiones, que se resuelve con el umbral.

**Lectura: el AUC es handoff, no punto de anclaje.** No hay un «valor verdadero» publicable del AUC (depende de la preparación: imputación, split, escalado); por eso el contraste es de **pertenencia a una banda** (0,83–0,85), no de Δ estricto. El operativo del venv es el que se reporta; la literatura (≈ 0,84–0,88 en el mismo dataset) es solo referencia (`INTERPRETACION_RESULTADOS.md Sección 7`).

**❓ Qué se quiere averiguar.** ¿Dónde conviene poner el punto de corte cuando perder un cliente cuesta S/300 y llamarlo por si acaso cuesta S/20?

- **La decisión concreta:** el umbral fija a cuánta gente contacta la campaña de retención. Es una decisión de política comercial, no un parámetro del modelo: el modelo es exactamente el mismo en todos los cortes, y su AUC tampoco cambia.
- **Antes de mirar el resultado:** los dos errores no cuestan igual — un falso negativo pesa **15 veces** más que un falso positivo—, de modo que el óptimo teórico es `t* = C_FP/(C_FP + C_FN) = 20/320 ≈ 0,06`, muy lejos del 0,5. Si la curva de costo tuviera su mínimo **en 0,5**, el corte por defecto quedaría justificado. Si el mínimo aparece **muy a la izquierda**, la lectura es que conviene sobre-avisar: se acepta llamar a mucha gente que no pensaba irse con tal de no perder a quien sí — la misma lógica de un tamizaje de bajo costo para una enfermedad grave.

🔎 **Qué hace este código.** Es el barrido de umbral del entregable.
- `C_FN, C_FP = 300, 20` — costos: un cliente perdido cuesta **15×** una llamada de retención.
- El bucle `for t in np.arange(0.1, 0.91, 0.1)` recorre umbrales de 0,1 a 0,9 y, en cada uno, calcula la matriz, `recall`, `precision` y el **costo total** `= C_FN·FN + C_FP·FP`.
- `tabla_umbral['costo'].idxmin()` localiza el umbral que **minimiza el costo**; `t* = C_FP/(C_FP+C_FN)` es el óptimo teórico.

In [ ]:
# Paso 4 — Elegir el umbral por costo (drill 2). Un FN cuesta MUCHO más que un FP.
C_FN, C_FP = 300, 20          # S/300 cliente perdido  vs  S/20 llamada de retención  ->  FN/FP = 15x
filas = []
for t in np.arange(0.1, 0.91, 0.1):
    tn, fp, fn, tp = confusion_matrix(yte, (prob_te >= t).astype(int)).ravel()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    costo = C_FN * fn + C_FP * fp
    filas.append([round(float(t), 2), tp, fp, fn, tn, round(precision, 3), round(recall, 3), int(costo)])
tabla_umbral = pd.DataFrame(filas, columns=["umbral", "VP", "FP", "FN", "VN", "precision", "recall", "costo"])

t_optimo = float(tabla_umbral.loc[tabla_umbral["costo"].idxmin(), "umbral"])
t_teorico = C_FP / (C_FP + C_FN)
print(f"Ratio de costo FN/FP = {C_FN}/{C_FP} = {C_FN//C_FP}x   |   umbral teórico t* = C_FP/(C_FP+C_FN) = {t_teorico:.3f}")
print(f"Umbral que MINIMIZA el costo en el barrido = {t_optimo}  (muy por debajo del 0,5 por defecto)")
tabla(tabla_umbral, "Barrido de umbral: matriz, recall, precisión y costo esperado = C_FN·FN + C_FP·FP")

📖 **Cómo leer esta salida.** Al **bajar** el umbral, `FN` cae (se pierden menos fugas) y `FP` sube (más falsas alarmas de menor costo); la columna `costo` primero baja y luego vuelve a subir: su **mínimo** marca el umbral recomendado (~0,1), muy por debajo del 0,5 por defecto. El `recall` sube hasta ~0,95: se capta casi toda la fuga.

💡 **Intuición.** El umbral teórico se obtiene al igualar costos: `t* = C_FP/(C_FP+C_FN) = 20/(20+300) ≈ 0,06`. La lectura: como equivocarse por un FN cuesta 15 veces más que por un FP, **conviene marcar en riesgo a mucha más gente**. Es la misma lógica de un examen médico de bajo costo para una enfermedad grave: se prefiere sobre-avisar antes que perder un caso.

**Lectura de negocio (el núcleo del entregable).** Como un cliente perdido (S/300) cuesta **~15×** una llamada de retención (S/20), el umbral óptimo `t* = 20/320 ≈ 0,06` cae muy por debajo de 0,5. En el barrido, **bajar el umbral a ≈ 0,10** eleva el **recall de ~0,58 a ~0,95** (se capta casi toda la fuga) al precio de más falsos positivos, y **minimiza el costo esperado**. La decisión no maximiza la exactitud, sino que **minimiza el costo del error** (`INTERPRETACION_RESULTADOS.md Sección 6`).

🔎 **Qué hace este código.** Toma del barrido las filas del umbral **0,5** (por defecto) y **0,1** (óptimo), y calcula el **ahorro** relativo de costo `(costo_0,5 − costo_0,1)/costo_0,5`. Arma una tabla que enfrenta recall, FN y costo esperado en ambos umbrales.

In [ ]:
# Contraste #4 — umbral óptimo (0,1) vs por defecto (0,5): costo, recall y ahorro
f05 = tabla_umbral[tabla_umbral["umbral"] == 0.5].iloc[0]
f01 = tabla_umbral[tabla_umbral["umbral"] == 0.1].iloc[0]
ahorro = (f05["costo"] - f01["costo"]) / f05["costo"] * 100
comp_umbral = pd.DataFrame({
    "umbral": [0.5, 0.1],
    "rol": ["por defecto (baseline)", "óptimo por costo"],
    "recall": [f05["recall"], f01["recall"]],
    "FN (fugas no vistas)": [int(f05["FN"]), int(f01["FN"])],
    "costo esperado (S/)": [int(f05["costo"]), int(f01["costo"])],
})
tabla(comp_umbral, f"Contraste #4 — umbral óptimo vs por defecto: el costo cae {ahorro:.0f}%  (recall {f05['recall']:.3f} -> {f01['recall']:.3f})")

📖 **Cómo leer esta salida.** La comparación es directa: pasar de 0,5 a 0,1 sube el `recall` (de ~0,58 a ~0,95), **reduce los FN** (fugas no vistas) y baja el `costo esperado` alrededor de **67 %**. El título reporta ese ahorro. Es el argumento central del entregable: la mejora no vino de cambiar el modelo, sino la **política de decisión**.

**Lectura: el umbral es política, no modelo.** El AUC **no cambia** al mover el umbral; lo que cambia es la **política de errores**. Pasar de 0,5 a 0,1 baja el costo esperado ~67 % y recupera casi toda la fuga, a cambio de más contactos. En la práctica se acota además por la **capacidad operativa** de la campaña. *El remuestreo/SMOTE para el desbalance es S10; aquí la palanca es el costo* (`DEFINICIONES.md Sección 14`).

**❓ Qué se quiere averiguar.** El modelo ya sabe **a quién** llamar. ¿Sabe también **qué ofrecerle**: qué rasgos del cliente empujan la fuga y cuáles la frenan?

- **La decisión concreta:** de aquí se obtiene el contenido de la campaña —migrar a contrato largo, reforzar el acompañamiento de los primeros meses, revisar la oferta de fibra— y no solo la lista de destinatarios. Sin esta lectura, el modelo entrega nombres pero ninguna palanca.
- **Antes de mirar el resultado:** cada coeficiente se lee como odds ratio. Un OR **> 1** señala un factor de riesgo, sobre el que se actúa para reducirlo; un OR **< 1** señala un factor protector, que se convierte en la palanca que conviene empujar. Si el contrato a dos años y la antigüedad resultan **por debajo de 1**, retienen; si la fibra óptica resulta **por encima**, es un segmento de riesgo. Atención a la unidad: en las variables numéricas el OR está por **una desviación estándar**, no por un mes ni por un sol, y en las dummies siempre frente a su categoría de referencia. Y el odds ratio expresa asociación, no causa: la campaña se valida después con un experimento A/B.

🔎 **Qué hace este código.** Lee los coeficientes del clasificador (`clf.coef_[0]`), los ordena por **magnitud absoluta** (`.abs().sort_values()`) y toma los 8 mayores; `np.exp(top)` los pasa a **odds ratio**. Para las numéricas (escaladas) el OR es por **1 desviación estándar**; para las dummies, frente a su categoría de referencia.

In [ ]:
# Paso 5 — Interpretar los coeficientes del churn en odds ratio (palancas de retención)
coef_churn = pd.Series(clf.coef_[0], index=Xt.columns)
top = coef_churn.reindex(coef_churn.abs().sort_values(ascending=False).index).head(8)
tabla_or_churn = pd.DataFrame({"coef": top.round(3), "odds_ratio": np.exp(top).round(3)})
tabla(tabla_or_churn, "Top-8 factores del churn (numéricas: OR por 1 desv. estándar; dummies: OR vs referencia)")

📖 **Cómo leer esta salida.** Un `odds_ratio > 1` es factor de **riesgo** de fuga; `< 1`, **protector**. Se espera ver `tenure` (antigüedad) con OR < 1 (los clientes veteranos se van menos), `Contract` a dos años con OR bajo (retiene), `InternetService = Fibra óptica` con OR alto (segmento de riesgo) y `TechSupport`/`OnlineSecurity` con OR < 1 (los servicios de valor agregado fidelizan). Son las **palancas de retención**.

**Recomendación de retención.** Los odds ratio coinciden con la evidencia del sector (las fuentes de actualidad de la sesión):

- **`tenure` (antigüedad): OR < 1** — cuanto más antiguo el cliente, mucho menor el riesgo. Los **primeros meses** son los críticos.
- **`Contract` a dos años: OR bajo** frente a *mes a mes*: **migrar a contratos largos** es la palanca más fuerte.
- **`InternetService = Fibra óptica`: OR alto** — segmento de **alto riesgo** (precio/expectativa): foco de campañas.
- **`TechSupport` / `OnlineSecurity`: OR < 1** — los servicios de valor agregado **retienen**.

**Acción sugerida:** priorizar clientes *mes a mes*, con **poca antigüedad** y **fibra óptica**, usando el **umbral bajo por costo**; validar el impacto real con un **experimento A/B (S02)**, ya que el OR es asociación, no causa.

<a id="sec-5"></a>

---

## Transversal — Exportación a Excel y figuras de resultados (Sección 5 del cuaderno)

> 💾 **Sección 5 de 9 — En esta sección:** se vuelca todo a `S09_resultados.xlsx` y se generan las figuras LEYENDO ese Excel.  —  [↑ índice](#indice)


Convención del curso: los resultados y pruebas del modelo se vuelcan a `resultados/S09_resultados.xlsx`, y las **figuras de resultados se generan LEYENDO ese Excel** (no desde objetos en memoria). El contrato de `logistica_saheart` (A1:B6) se mantiene intacto; se añade **B7 = prevalencia de churn** para que el dato de apertura de negocio también se lea del Excel («cero literales numéricos» en el deck).

🔎 **Qué hace este código.** Crea el libro de Excel con `openpyxl.Workbook()` y escribe **5 hojas**: `logistica_saheart` (contrato A1:B7 con los coeficientes, el AUC, la deviance/gl y la prevalencia), `odds_ratios`, `matriz_confusion_umbral`, `glm_poisson` y la auxiliar `roc_churn` (los puntos de la curva). `wb.save(XLSX)` lo guarda. La convención del curso es **escribir aquí y leer desde aquí** para las figuras y el tablero, de modo que nada se muestre desde memoria.

In [ ]:
# Construir el Excel con las 4 hojas del contrato + 1 hoja auxiliar para la curva ROC
from openpyxl import Workbook
wb = Workbook()

# --- Hoja 1: logistica_saheart (CONTRATO A1:B6 intacto; + B7 prevalencia; valores CALCULADOS) ---
ws = wb.active; ws.title = "logistica_saheart"
ws["A1"] = "metrica"; ws["B1"] = "valor"
ws["A2"] = "coef_tobacco";               ws["B2"] = round(coef_tobacco, 5)
ws["A3"] = "coef_famhist";               ws["B3"] = round(coef_famhist, 5)
ws["A4"] = "coef_age";                   ws["B4"] = round(coef_age, 5)
ws["A5"] = "auc_telco_churn";            ws["B5"] = round(auc_churn, 4)
ws["A6"] = "poisson_deviance_df_ships";  ws["B6"] = round(dev_gl_ships, 4)
ws["A7"] = "prevalencia_churn";          ws["B7"] = round(prevalencia_churn, 4)

# --- Hoja 2: odds_ratios (por predictor de SAheart: coef, OR, p, IC del OR) ---
ws2 = wb.create_sheet("odds_ratios")
ws2.append(["predictor", "coef", "odds_ratio", "p_valor", "or_ic_bajo", "or_ic_alto"])
ci = logit.conf_int()
for nombre in logit.params.index:
    ws2.append([nombre, round(float(logit.params[nombre]), 5),
                round(float(np.exp(logit.params[nombre])), 4),
                round(float(logit.pvalues[nombre]), 4),
                round(float(np.exp(ci.loc[nombre, 0])), 4),
                round(float(np.exp(ci.loc[nombre, 1])), 4)])

# --- Hoja 3: matriz_confusion_umbral (CM a 0,5 + tabla de umbral por costo) ---
ws3 = wb.create_sheet("matriz_confusion_umbral")
tn, fp, fn, tp = cm05.ravel()
ws3.append(["matriz_confusion_umbral_0.5", "pred_se_queda", "pred_se_fuga"])
ws3.append(["real_se_queda", int(tn), int(fp)])
ws3.append(["real_se_fuga", int(fn), int(tp)])
ws3.append([])
ws3.append(["umbral", "VP", "FP", "FN", "VN", "precision", "recall", "costo"])
for _, f in tabla_umbral.iterrows():
    ws3.append([float(f["umbral"]), int(f["VP"]), int(f["FP"]), int(f["FN"]), int(f["VN"]),
                float(f["precision"]), float(f["recall"]), int(f["costo"])])

# --- Hoja 4: glm_poisson (ships: coef, RR, deviance, gl, deviance/gl) ---
ws4 = wb.create_sheet("glm_poisson")
ws4.append(["termino", "coef", "rate_ratio", "p_valor"])
for nombre in pois_ships.params.index:
    ws4.append([nombre, round(float(pois_ships.params[nombre]), 5),
                round(float(np.exp(pois_ships.params[nombre])), 4),
                round(float(pois_ships.pvalues[nombre]), 4)])
ws4.append([])
ws4.append(["deviance", round(dev_ships, 4)])
ws4.append(["gl", gl_ships])
ws4.append(["deviance_gl", round(dev_gl_ships, 4)])

# --- Hoja auxiliar: roc_churn (puntos de la curva ROC, para la figura de resultados) ---
ws5 = wb.create_sheet("roc_churn")
ws5.append(["fpr", "tpr"])
for a, b in zip(fpr, tpr):
    ws5.append([round(float(a), 5), round(float(b), 5)])

wb.save(XLSX)
print("Excel guardado en:", XLSX)
print("Hojas:", wb.sheetnames)
print(f"Contrato logistica_saheart -> B2={round(coef_tobacco,5)}, B3={round(coef_famhist,5)}, "
      f"B4={round(coef_age,5)}, B5={round(auc_churn,4)}, B6={round(dev_gl_ships,4)}, B7={round(prevalencia_churn,4)}")

📖 **Cómo leer esta salida.** El `print` confirma la ruta, la lista de **5 hojas** y los valores del contrato `logistica_saheart`: `B2` (coef_tobacco), `B3` (coef_famhist), `B4` (coef_age), `B5` (auc_telco_churn), `B6` (poisson_deviance_df_ships) y `B7` (prevalencia_churn). A partir de aquí, cada figura y el tablero final **releen** estas celdas: si un número no coincide con el Excel, es un error detectable.

### Figuras de resultados (leídas del Excel) — transversal (subsección 5.1)

🔎 **Qué hace este código.** **Lee** del Excel los puntos de la curva (hoja `roc_churn`) y el AUC (hoja `logistica_saheart`) con `pd.read_excel`, y traza la **ROC**: sensibilidad (eje Y) frente a tasa de falsos positivos (eje X). La diagonal punteada es el azar (AUC 0,50). Nada se toma de memoria: la figura reconstruye el resultado desde el archivo.

In [ ]:
# Figura 1 — Curva ROC del churn (leída de la hoja roc_churn; AUC de logistica_saheart)
roc = pd.read_excel(XLSX, sheet_name="roc_churn").dropna(subset=["fpr", "tpr"])
meta = pd.read_excel(XLSX, sheet_name="logistica_saheart")
auc_leido = float(meta.loc[meta["metrica"] == "auc_telco_churn", "valor"].iloc[0])
fig, ax = plt.subplots(figsize=(5.2, 5.0))
ax.plot(roc["fpr"], roc["tpr"], color=UPC_ROJO, lw=2.2, label=f"Logística (AUC = {auc_leido:.3f})")
ax.plot([0, 1], [0, 1], ls="--", color=UPC_GRIS, lw=1, label="Azar (AUC = 0,50)")
ax.set_xlabel("Tasa de falsos positivos (1 − especificidad)")
ax.set_ylabel("Sensibilidad (recall)")
ax.set_title("Curva ROC — modelo de fuga de clientes (test)")
ax.legend(loc="lower right", frameon=False)
mostrar(fig, FIGURAS / "roc_churn.png")

**Lectura.** La curva **se despega de la diagonal** (azar): el modelo separa fugas de permanencias muy por encima del 0,5. Pero la ROC recorre **todos** los umbrales a la vez; **no** dice cuál usar — esa decisión es del umbral por costo (`INTERPRETACION_RESULTADOS.md Sección 7`).

🔎 **Qué hace este código.** Igual que la ROC anterior, pero además localiza en el barrido de umbral la fila de **mínimo costo** y marca sobre la curva el **punto de operación** `t*`: calcula su sensibilidad `VP/(VP+FN)` y su tasa de falsos positivos `FP/(FP+VN)`, y los dibuja con un marcador y una anotación. Traduce visualmente 'qué punto de la ROC se eligió y por qué'.

In [ ]:
# Figura — ROC del churn con el PUNTO DE OPERACIÓN elegido por costo (t*), leído del Excel
roc = pd.read_excel(XLSX, sheet_name="roc_churn").dropna(subset=["fpr", "tpr"])
crudo = pd.read_excel(XLSX, sheet_name="matriz_confusion_umbral", header=None)
ini = crudo.index[crudo[0] == "umbral"][0]
sweep = crudo.iloc[ini + 1:].copy()
sweep.columns = crudo.iloc[ini].values
sweep = sweep.dropna(subset=["umbral"]).astype(
    {"umbral": float, "VP": float, "FP": float, "FN": float, "VN": float, "costo": float})
fila = sweep.loc[sweep["costo"].idxmin()]
tpr_op = fila["VP"] / (fila["VP"] + fila["FN"])           # sensibilidad en t*
fpr_op = fila["FP"] / (fila["FP"] + fila["VN"])           # 1 − especificidad en t*
meta = pd.read_excel(XLSX, sheet_name="logistica_saheart")
auc_leido = float(meta.loc[meta["metrica"] == "auc_telco_churn", "valor"].iloc[0])

fig, ax = plt.subplots(figsize=(5.4, 5.0))
ax.plot(roc["fpr"], roc["tpr"], color=UPC_ROJO, lw=2.2, label=f"Logística (AUC = {auc_leido:.3f})")
ax.plot([0, 1], [0, 1], ls="--", color=UPC_GRIS, lw=1, label="Azar (AUC = 0,50)")
ax.scatter([fpr_op], [tpr_op], color=UPC_TINTA, s=130, zorder=5,
           label=f"Punto de operación t* = {fila['umbral']:.1f} (recall = {tpr_op:.2f})")
ax.annotate("umbral por costo\n(FN 15× FP)", xy=(fpr_op, tpr_op),
            xytext=(min(fpr_op + 0.20, 0.72), max(tpr_op - 0.22, 0.10)), fontsize=9, color=UPC_TINTA,
            arrowprops=dict(arrowstyle="->", color=UPC_TINTA))
ax.set_xlabel("Tasa de falsos positivos (1 − especificidad)")
ax.set_ylabel("Sensibilidad (recall)")
ax.set_title("ROC del churn con el punto de operación elegido por costo")
ax.legend(loc="lower right", frameon=False, fontsize=8.5)
mostrar(fig, FIGURAS / "S09_fig_roc_punto_operacion.png")

🔎 **Qué hace este código.** Lee la hoja `odds_ratios` y dibuja un **forest plot**: cada predictor es un punto en su odds ratio, con una barra horizontal (`ax.errorbar`) que representa el **IC 95 %**. La línea vertical en 1 marca 'sin efecto'; se colorean en rojo los predictores con `p < 0,05` (significativos). Si la barra **cruza 1**, el efecto no es distinguible de cero.

In [ ]:
# Figura 2 — Odds ratio de SAheart con IC 95% (forest plot, leído de odds_ratios)
orx = pd.read_excel(XLSX, sheet_name="odds_ratios")
orx = orx[orx["predictor"] != "const"].sort_values("odds_ratio")
signif = orx["p_valor"] < 0.05
colores = [UPC_ROJO if s else UPC_GRIS for s in signif]
fig, ax = plt.subplots(figsize=(6.4, 4.2))
yy = np.arange(len(orx))
ax.errorbar(orx["odds_ratio"], yy,
            xerr=[orx["odds_ratio"] - orx["or_ic_bajo"], orx["or_ic_alto"] - orx["odds_ratio"]],
            fmt="none", ecolor=UPC_GRIS, elinewidth=1.5, capsize=3)
ax.scatter(orx["odds_ratio"], yy, color=colores, s=45, zorder=3)
ax.axvline(1.0, ls="--", color=UPC_TINTA, lw=1)
ax.set_yticks(yy); ax.set_yticklabels(orx["predictor"])
ax.set_xlabel("Odds ratio = exp(β)   (línea en 1 = sin efecto)")
ax.set_title("Factores de riesgo de cardiopatía (rojo = significativo, p < 0,05)")
mostrar(fig, FIGURAS / "odds_ratio_saheart.png")

**Lectura: quién es factor de riesgo.** `famhist` (OR ≈ 2,56) es el factor más fuerte y su **IC no cruza 1** (significativo, en rojo); `age`, `ldl` y `tobacco` también. En cambio `sbp`, `obesity` y `alcohol` tienen un **IC del OR que cruza 1** = sin efecto detectable (`INTERPRETACION_RESULTADOS.md Sección 2`). El OR es asociación condicional, no causa (Sección 8).

🔎 **Qué hace este código.** Lee de `matriz_confusion_umbral` la tabla de barrido y grafica el **costo esperado frente al umbral**; marca con un punto el **mínimo** y con una línea punteada el 0,5 por defecto. La curva en forma de U hace visible que el óptimo está muy a la izquierda del 0,5.

In [ ]:
# Figura 3 — Costo esperado vs umbral (leído de matriz_confusion_umbral)
crudo = pd.read_excel(XLSX, sheet_name="matriz_confusion_umbral", header=None)
inicio = crudo.index[crudo[0] == "umbral"][0]
tab = crudo.iloc[inicio + 1:].copy()
tab.columns = crudo.iloc[inicio].values
tab = tab.dropna(subset=["umbral"]).astype({"umbral": float, "costo": float, "recall": float})
fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.plot(tab["umbral"], tab["costo"], "-o", color=UPC_ROJO, lw=2, label="Costo esperado (S/)")
jmin = tab["costo"].idxmin()
ax.scatter([tab.loc[jmin, "umbral"]], [tab.loc[jmin, "costo"]], color=UPC_TINTA, s=120, zorder=5,
           label=f"Mínimo en umbral = {tab.loc[jmin, 'umbral']:.1f}")
ax.axvline(0.5, ls="--", color=UPC_GRIS, lw=1, label="Umbral por defecto (0,5)")
ax.set_xlabel("Umbral de decisión")
ax.set_ylabel("Costo esperado  =  C_FN·FN + C_FP·FP")
ax.set_title("El umbral que minimiza el costo está muy por debajo de 0,5")
ax.legend(frameon=False)
mostrar(fig, FIGURAS / "costo_vs_umbral.png")

**Lectura.** El mínimo de costo cae en **0,1 ≪ 0,5**: con un FN 15× más caro que un FP, conviene **marcar en riesgo a más gente**. Del score a la acción: traducir los OR a **palancas de retención** (contrato, antigüedad, soporte) y actuar sobre los clientes que el umbral bajo señala (`INTERPRETACION_RESULTADOS.md Sección 6`).

### Figuras de EDA (cálculo directo) — transversal (subsección 5.2)

🔎 **Qué hace este código.** Dibuja la matriz de confusión al 0,5 como **mapa de calor** (`ax.imshow(cm05, cmap='Reds')`): cuanto más oscura la celda, más casos. Anota en cada celda su etiqueta (VN/FP/FN/VP) y su conteo. Es la misma matriz de la Sección 4, ahora en forma visual.

In [ ]:
# Figura 4 — Matriz de confusión del churn como heatmap (umbral 0,5)
fig, ax = plt.subplots(figsize=(4.6, 4.0))
im = ax.imshow(cm05, cmap="Reds")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred: se queda", "Pred: se fuga"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Real: se queda", "Real: se fuga"])
etiquetas = np.array([["VN", "FP"], ["FN", "VP"]])
for i in range(2):
    for j in range(2):
        color = "white" if cm05[i, j] > cm05.max() / 2 else UPC_TINTA
        ax.text(j, i, f"{etiquetas[i, j]}\n{cm05[i, j]}", ha="center", va="center",
                color=color, fontsize=12, fontweight="bold")
ax.set_title("Matriz de confusión — churn (umbral 0,5)")
ax.grid(False)
mostrar(fig, FIGURAS / "matriz_confusion_churn.png")

🔎 **Qué hace este código.** Cuenta las clases del target `chd` con `value_counts()` y las grafica como barras, anotando el porcentaje de cada una. Sirve como EDA: muestra el grado de **desbalance** del problema de cardiopatía (~35 % positivos).

In [ ]:
# Figura 5 — Distribución del target chd en SAheart (EDA)
conteo = sa["chd"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.bar(["Sin cardiopatía (0)", "Con cardiopatía (1)"], conteo.values,
       color=[UPC_GRIS, UPC_ROJO])
for i, v in enumerate(conteo.values):
    ax.text(i, v + 3, f"{v}  ({v/conteo.sum()*100:.1f}%)", ha="center", fontsize=10)
ax.set_ylabel("Número de pacientes")
ax.set_title("Distribución del target chd (SAheart, n = 462)")
ax.grid(axis="x")
mostrar(fig, FIGURAS / "distribucion_chd.png")

**Lectura (EDA).** La matriz al 0,5 hace visible el problema: 159 FN frente a 215 VP (recall ~0,58). Y `chd` está algo desbalanceado (~35 % positivos), lo que —como en churn— vuelve engañosa la exactitud y obliga a leer recall/precisión y a razonar sobre el costo del error.

🔎 **Qué hace este código.** Construye con `pandas.plotting.scatter_matrix` la **matriz de dispersión** de los **7 factores de riesgo** de SAheart (`sbp`, `tobacco`, `ldl`, `famhist`, `obesity`, `alcohol`, `age`), coloreando cada punto según `chd` (**rojo** = con cardiopatía, **gris** = sin). En la diagonal van los histogramas de cada variable. Es la réplica visual de la **Figura 4.12 de ESL (p. 122)**; no toca el Excel.

In [ ]:
# Figura 6 — Matriz de dispersión de los 7 factores de riesgo de SAheart, coloreada por chd
# Reproduce ESL Figura 4.12 (p. 122): scatterplot matrix del South African Heart Disease data.
from pandas.plotting import scatter_matrix

colores_chd = np.where(sa["chd"].values == 1, UPC_ROJO, UPC_GRIS)   # rojo = caso (con cardiopatía)
ejes_sm = scatter_matrix(
    sa[predictores], c=colores_chd, diagonal="hist",
    figsize=(11, 11), alpha=0.55, s=10,
    hist_kwds={"color": UPC_TINTA, "bins": 18},
)
for fila_sm in ejes_sm:
    for ax_sm in fila_sm:
        ax_sm.grid(False)
        ax_sm.tick_params(labelsize=7)
        ax_sm.xaxis.label.set_size(9); ax_sm.yaxis.label.set_size(9)
fig_sm = ejes_sm[0, 0].get_figure()
fig_sm.suptitle(
    "Matriz de dispersion de SAheart coloreada por chd (rojo = con cardiopatia)\n"
    "Reproduce ESL Figura 4.12 (p. 122)",
    fontsize=13, y=1.0,
)
mostrar(fig_sm, FIGURAS / "S09_fig_scatter_matrix_esl412.png")

📄 **En el paper.** Esta figura reproduce la **Figura 4.12 (p. 122)** de **Hastie, Tibshirani & Friedman (2009), _The Elements of Statistical Learning_, 2.ª ed. (Sección 4.4.2)**: la matriz de dispersión del South African Heart Disease data con casos y controles coloreados (rojo = caso). `famhist`, por ser binaria (Present/Absent → 1/0), aparece como dos bandas horizontales, igual que en el original. La lectura coincide con la del libro: los casos (rojo) se concentran hacia los valores altos de `age`, `tobacco`, `ldl` y en `famhist = Present`, coherente con los coeficientes positivos de la Tabla 4.2 replicados en la Sección 2.

<a id="sec-6"></a>

---

## Transversal — Tablero consolidado (leído del Excel) (Sección 6 del cuaderno)

> ✅ **Sección 6 de 9 — En esta sección:** se re-leen las celdas del Excel y se arma la tabla-contraste final de los cinco targets.  —  [↑ índice](#indice)


El cierre **re-lee** las celdas de `logistica_saheart` del Excel y arma la tabla-contraste final de los cinco targets: certifica que **lo que se muestra es exactamente lo que se guardó** (obtenido leído del Excel, no de memoria).

🔎 **Qué hace este código.** Cierra el ciclo: en vez de usar los valores en memoria, **relee** las celdas del Excel con el ayudante `leer_celda('logistica_saheart', 'B2'...)` y las pasa a `contraste(...)`. Así certifica que lo que se muestra en el tablero es **exactamente** lo que quedó guardado en el archivo.

In [ ]:
# Tablero consolidado — se RELEEN los valores del Excel (obtenido leído del Excel)
tablero = contraste([
    {"concepto": "coef tobacco",      "obtenido": leer_celda("logistica_saheart", "B2"), "esperado": 0.080, "tol": 0.015, "fuente": "ESL Tabla 4.2"},
    {"concepto": "coef famhist",      "obtenido": leer_celda("logistica_saheart", "B3"), "esperado": 0.939, "tol": 0.03,  "fuente": "ESL Tabla 4.2"},
    {"concepto": "coef age",          "obtenido": leer_celda("logistica_saheart", "B4"), "esperado": 0.043, "tol": 0.01,  "fuente": "ESL Tabla 4.2"},
    {"concepto": "AUC churn (test)",  "obtenido": leer_celda("logistica_saheart", "B5"), "banda": (0.83, 0.85), "fuente": "banda literatura (handoff)"},
    {"concepto": "ships deviance/gl", "obtenido": leer_celda("logistica_saheart", "B6"), "esperado": 1.55,  "tol": 0.15,  "fuente": "McCullagh & Nelder"},
])
prev = leer_celda("logistica_saheart", "B7")
tabla(tablero, f"Tablero de réplica S09 — 5 targets leídos del Excel vs benchmark  (prevalencia de churn en Excel: {prev*100:.1f}%)")

📖 **Cómo leer esta salida.** Es el **tablero de certificación**: cinco targets (tres coeficientes logísticos, el AUC del churn y la deviance/gl de ships), cada uno con su `obtenido` **leído del Excel**, su benchmark y el veredicto ✔/✗. Todos en ✔ y la prevalencia leída de `B7` confirman que la réplica es fiel y trazable de punta a punta.

📄 **En el paper (exactitud fila a fila).** Los tres coeficientes logísticos del tablero coinciden **exactamente, dentro de tolerancia**, con los publicados en **ESL Tabla 4.2 (Sección 4.4.2, p. 122)**: `tobacco` = 0,080; `famhist[Present]` = 0,939; `age` = 0,043 (y `ldl` = 0,185, contrastado en la Sección 2). La `deviance/gl` de `ships` reproduce la referencia de **McCullagh & Nelder (1989), Sección 6.3.2** (≈ 38,7/25 gl → φ ≈ 1,55). El valor **operativo** del venv y el **benchmark** publicado concuerdan fila por fila.

**Conclusión de negocio.** La réplica queda **verificada**: los coeficientes logísticos reproducen ESL Tabla 4.2, el GLM Poisson de `ships` muestra sobre-dispersión leve y el clasificador de churn ordena dentro de la banda de literatura. Pero el valor de negocio no está en el AUC, sino en la **decisión**: con un FN 15× más caro que un FP, mover el umbral de 0,5 a ~0,1 recupera casi toda la fuga y **baja el costo esperado ~67 %**. El modelo no cambia; cambia la **política**. Las palancas (contrato, antigüedad, soporte) se validan luego con un **A/B (S02)**.

<a id="verif-base"></a>
### ✅ Verificación desde la base (recomputado ≈ paper ≈ Excel) — capítulo 9.7 (subsección 6.1)

✅ **Estos números NO están transcritos manualmente: son producto de EJECUTAR el modelo sobre la base.** Aquí se cierra el círculo del curso reflejando —visible y explicada— la misma lógica de el material de referencia de la sesión: se **recomputan** los coeficientes, la sobre-dispersión y el AUC **desde los datos ya cargados** (`sa`, `sh`, `yte`/`prob_te`) y se arma una tabla de tres columnas **recomputado (venv) | ESL/paper | Excel**. Cada fila se cierra con un `assert` de que *recomputado ≈ Excel*: si el Excel se hubiera editado manualmente o quedado obsoleto, esta celda **fallaría**. La cadena es: **los datos producen los números → los números se escriben al Excel → el deck lee el Excel.** Nada se teclea.


In [ ]:
# ✅ Recomputar desde la base (misma lógica que el material de referencia de la sesión, aquí visible y explicada)

# (1) Coeficientes logísticos: re-ajustar el Logit de 7 predictores sobre 'sa' (ya cargado)
vb_logit = sm.Logit(sa["chd"], sm.add_constant(sa[predictores])).fit(disp=0)
vb_tobacco = float(vb_logit.params["tobacco"])
vb_famhist = float(vb_logit.params["famhist"])
vb_age     = float(vb_logit.params["age"])

# (2) Sobre-dispersión de ships: deviance/gl del GLM Poisson (recomputado del ajuste sobre 'sh')
vb_dev_gl = float(pois_ships.deviance / pois_ships.df_resid)

# (3) AUC del churn: recomputado sobre las predicciones del test
vb_auc = float(roc_auc_score(yte, prob_te))

# --- Excel (valores LEÍDOS del archivo, no de memoria) y paper/ESL (benchmark publicado, ETIQUETADO) ---
xls = {
    "coef_tobacco (SAheart)": float(leer_celda("logistica_saheart", "B2")),
    "coef_famhist (SAheart)": float(leer_celda("logistica_saheart", "B3")),
    "coef_age (SAheart)":     float(leer_celda("logistica_saheart", "B4")),
    "AUC churn (Telco)":      float(leer_celda("logistica_saheart", "B5")),
    "deviance/gl (ships)":    float(leer_celda("logistica_saheart", "B6")),
}
recomputado = {
    "coef_tobacco (SAheart)": vb_tobacco,
    "coef_famhist (SAheart)": vb_famhist,
    "coef_age (SAheart)":     vb_age,
    "AUC churn (Telco)":      vb_auc,
    "deviance/gl (ships)":    vb_dev_gl,
}
paper = {
    "coef_tobacco (SAheart)": "0,080 (ESL Tabla 4.2)",
    "coef_famhist (SAheart)": "0,939 (ESL Tabla 4.2)",
    "coef_age (SAheart)":     "0,043 (ESL Tabla 4.2)",
    "AUC churn (Telco)":      "0,83-0,85 (banda literatura)",
    "deviance/gl (ships)":    "≈1,55 (McCullagh & Nelder 6.3.2)",
}
verif = pd.DataFrame({
    "recomputado (venv)":       {k: round(recomputado[k], 5) for k in recomputado},
    "ESL / paper (benchmark)":  paper,
    "Excel (leído)":            {k: round(xls[k], 5) for k in xls},
})
tabla(verif, "Verificación desde la base: recomputado (venv) | ESL/paper | Excel — tobacco/famhist/age, deviance/gl y AUC")

# assert: recomputado ≈ Excel (mismas tolerancias que el validador de QA)
for k, tol in [("coef_tobacco (SAheart)", 1e-3), ("coef_famhist (SAheart)", 1e-3),
               ("coef_age (SAheart)", 1e-3), ("AUC churn (Telco)", 1.5e-2), ("deviance/gl (ships)", 1e-2)]:
    assert abs(recomputado[k] - xls[k]) <= tol, (k, recomputado[k], xls[k])
print("✔ recomputado ≈ Excel en las 5 filas: el Excel es producto de la ejecución, no un registro tecleado.")


📖 **Cómo leer.** Las tres columnas cuentan la misma historia por tres caminos: **recomputado** (lo que resulta de ejecutar el modelo ahora), **ESL/paper** (lo publicado, como benchmark etiquetado) y **Excel** (lo que quedó guardado y alimenta al deck). Que coincidan —y que el `assert` no falle— certifica la **integridad de la réplica**: cada cifra es un resultado de ejecución reproducible, no un dato copiado. Es exactamente lo que el material de referencia de la sesión comprueba desde fuera; aquí el alumno lo ve por dentro.


<a id="sec-7"></a>

---

## 9.8 en profundidad — Laboratorio por industria — el mismo método en cinco sectores (Sección 7 del cuaderno)

> 🏭 **Sección 7 de 9 — En esta sección:** se transfiere el MISMO método de la sesión (regresión **logística**, **umbral por costo** y **GLM Poisson**) a cinco sectores distintos; el objetivo es ver que la herramienta es la misma aunque cambie el negocio.  —  [↑ índice](#indice)

### Objetivo
Mostrar que la logística/GLM y el umbral por costo no son "del churn": son un patrón **transferible**. Cada mini-caso sigue el mismo guion — **contexto de negocio → qué modelar → qué decisión**.

### Cómo leer esta sección
- **Resuelto** = se ejecuta aquí reutilizando un modelo ya ajustado en la sesión (no recalcula nada del Excel).
- **Tarea guiada del alumno** = enunciado listo para resolver con un dataset abierto; no se ejecuta en el cuaderno.

| # | Industria | Dato del curso | Qué se modela | Qué decisión | Estado |
|---|---|---|---|---|:--:|
| 1 | **Banca** — riesgo de crédito | — (dataset abierto) | P(default) ~ ingresos, deuda, historial | a quién aprobar; umbral por costo del impago | Tarea guiada |
| 2 | **Salud** — cardiopatía | SAheart (Sección 2) | P(chd) ~ 7 factores de riesgo | a quién priorizar para tamizaje | **Resuelto** |
| 3 | **Telecom** — fuga de clientes | Telco (Sección 4) | P(churn) ~ contrato, cargos, servicios | a quién contactar para retener | **Resuelto** |
| 4 | **Retail / e-commerce** — conversión | — (dataset abierto) | P(compra) ~ navegación, origen, recencia | a quién mostrar un cupón | Tarea guiada |
| 5 | **Manufactura / seguros** — incidentes | ships (Sección 3) | tasa de incidentes ~ tipo/año (GLM Poisson) | qué activos inspeccionar primero | **Resuelto** |

### Caso 1, Banca — riesgo de crédito (default) ,  *Tarea guiada del alumno*

**Contexto.** Un banco debe decidir a qué solicitantes aprobar un préstamo; conceder a quien no pagará (default) cuesta el saldo impago, mucho más que rechazar a un buen cliente.
**Qué modelar.** Una **regresión logística** de `default` (1/0) sobre ingreso, ratio deuda/ingreso, antigüedad laboral e historial de mora; interpretar los coeficientes como **odds ratio**.
**Qué decisión.** Fijar el **umbral por costo** con `C_FN` = saldo impago esperado y `C_FP` = margen perdido al rechazar; aprobar solo por debajo del score de riesgo elegido.
**Tarea guiada.** Descargar un dataset abierto de crédito (p. ej. UCI *Default of Credit Card Clients* o *Statlog German Credit*), ajustar el `Logit`, reportar el OR de cada factor y elegir el umbral que minimiza el costo esperado. *No hay dataset de banca en S09: se resuelve replicando el patrón de la Sección 2 y Sección 4.*

🔎 **Qué hace este código.** **Reutiliza** el `logit` ya ajustado en la Sección 2 (no recalcula): `logit.predict(X_sa)` da la probabilidad de cardiopatía de cada paciente, fija el umbral en la **prevalencia observada** (`y_sa.mean()`) y cuenta a cuántos priorizaría para tamizaje. `np.exp(logit.params['famhist'])` recupera el OR del factor de riesgo #1.

In [ ]:
# CASO 2 — SALUD (RESUELTO) — priorización de tamizaje coronario reutilizando el Logit de la Sección 2 (SAheart)
prob_chd = logit.predict(X_sa)                      # P(chd) del modelo ya ajustado (in-sample, ilustrativo)
umbral_tamizaje = float(y_sa.mean())                # regla simple de negocio: umbral = prevalencia observada
priorizados = int((prob_chd >= umbral_tamizaje).sum())
or_famhist = float(np.exp(logit.params["famhist"]))
tabla(pd.DataFrame({
    "concepto": ["pacientes en la muestra", "prevalencia de chd", "umbral de tamizaje (=prevalencia)",
                 "pacientes priorizados", "OR de famhist (factor de riesgo #1)"],
    "valor": [len(sa), f"{y_sa.mean()*100:.1f}%", f"{umbral_tamizaje:.2f}",
              priorizados, f"{or_famhist:.2f}x"],
}), "Caso salud (resuelto): a quién priorizar para tamizaje coronario")

📖 **Cómo leer esta salida.** La tabla resume la **regla de priorización**: sobre 462 pacientes, con el umbral igual a la prevalencia, el modelo marca para tamizaje temprano a los de riesgo estimado superior; `famhist` (OR ≈ 2,56) es la palanca de segmentación más fuerte. Es el mismo Logit de la réplica, ahora leído como **decisión clínica**.

**Decisión (salud).** Con el umbral fijado en la prevalencia observada, el modelo prioriza para tamizaje temprano a los pacientes cuyo riesgo estimado la supera (ver la salida anterior); `famhist` (OR ≈ 2,56) es la palanca de segmentación más fuerte. Es el **mismo Logit de la Sección 2**, ahora leído como **regla de priorización clínica**, no como réplica académica.

🔎 **Qué hace este código.** **Reutiliza** la logística de churn de la Sección 4 y su umbral por costo `t*`: clasifica el test con `prob_te >= t_optimo`, cuenta a los **marcados para retención** y calcula qué fracción de las fugas reales capta (recall). No reentrena nada.

In [ ]:
# CASO 3 — TELECOM (RESUELTO) — campaña de retención al umbral por costo reutilizando la logística de la Sección 4 (Telco)
t_ret = t_optimo                                    # umbral por costo hallado en Sección 4 (FN = 15x FP)
pred_ret = (prob_te >= t_ret).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred_ret).ravel()
marcados = int(pred_ret.sum())
tabla(pd.DataFrame({
    "concepto": ["clientes en test", "umbral por costo t*", "marcados para retención",
                 "fugas realmente captadas (recall)", "AUC del modelo"],
    "valor": [len(yte), f"{t_ret:.2f}", marcados, f"{tp/(tp+fn)*100:.0f}%", f"{auc_churn:.3f}"],
}), "Caso telecom (resuelto): a quién contactar para retener")

📖 **Cómo leer esta salida.** Al umbral por costo (muy por debajo de 0,5) la campaña marca a bastantes clientes y **capta la gran mayoría de las fugas** (recall alto) a cambio de falsos positivos de bajo costo. Es la política de la Sección 4 expresada como decisión de retención lista para operar.

**Decisión (telecom).** Al umbral por costo `t*` (muy por debajo de 0,5), la campaña contacta a los clientes marcados y **capta la gran mayoría de las fugas** (recall alto) a cambio de más falsos positivos de bajo costo. Es exactamente la política de la Sección 4, aquí resumida como decisión de retención.

### Caso 4, Retail / e-commerce — conversión ,  *Tarea guiada del alumno*

**Contexto.** Un e-commerce quiere anticipar qué visitante **convertirá** (compra) en la sesión actual para decidir a quién mostrar un cupón; el cupón cuesta margen, no mostrarlo a quien iba a comprar no cuesta nada.
**Qué modelar.** Una **logística** de `compra` (1/0) sobre páginas vistas, tiempo en sitio, recencia y canal de origen; OR por variable.
**Qué decisión.** Umbral por costo con `C_FN` = margen de la venta perdida y `C_FP` = costo del cupón; mostrar cupón solo por encima del umbral.
**Tarea guiada.** Usar un dataset abierto (p. ej. UCI *Online Shoppers Purchasing Intention*). *Online Retail II (S07/S08/S14) es transaccional y no trae una etiqueta de conversión por sesión, por eso este caso queda como tarea.*

🔎 **Qué hace este código.** **Reutiliza** el GLM Poisson de `ships` (Sección 3): `np.exp(pois_ships.params)` convierte cada coeficiente en **rate ratio**, descarta el intercepto y los ordena de mayor a menor. Los términos con rate ratio alto son los que concentran más incidentes por mes de servicio.

In [ ]:
# CASO 5 — MANUFACTURA/SEGUROS (RESUELTO) — priorizar inspección por tasa de incidentes reutilizando el GLM Poisson de la Sección 3 (ships)
rate_ratios = np.exp(pois_ships.params).drop("Intercept").sort_values(ascending=False)
tabla(pd.DataFrame({
    "término (tipo/año/periodo)": rate_ratios.index,
    "rate_ratio = exp(β)": rate_ratios.round(3).values,
}), "Caso seguros (resuelto): factores ordenados por tasa de incidentes (rate ratio)")
print(f"Diagnóstico: deviance/gl = {dev_gl_ships:.2f} (sobre-dispersión leve) -> los rate ratios son utilizables para priorizar.")

📖 **Cómo leer esta salida.** Los términos con **rate ratio > 1** tienen más incidentes por mes de servicio: son los tipos/periodos que una aseguradora o un astillero **inspecciona o tarifica primero**. El `offset = log(service)` garantiza comparar **tasas** (no conteos brutos), y la sobre-dispersión leve (φ ≈ 1,55) mantiene válida la lectura.

**Decisión (manufactura/seguros).** Los términos con **rate ratio > 1** concentran más incidentes por mes de servicio: son los tipos/periodos que una aseguradora o un astillero **inspecciona o tarifica primero**. El `offset = log(service)` asegura comparar **tasas** (no conteos brutos), y la sobre-dispersión leve (φ ≈ 1,55) mantiene válida la lectura.

---
**Síntesis de la sección.** Cambia el sector, cambia la variable respuesta (fuga, cardiopatía, incidentes), pero **el método es el mismo**: ajustar un GLM (logística o Poisson), leer los coeficientes como **odds/rate ratios** y convertir la probabilidad/tasa en una **decisión por costo**. Esa transferencia es el objetivo de aprendizaje de la sesión.  —  [↑ índice](#indice)


<a id="sec-8"></a>

---

## 9.3 en profundidad — Construcción del pipeline de churn desde cero (Sección 8 del cuaderno)

> 🧱 **Sección 8 de 9 — En esta sección:** el alumno **arma el clasificador de churn paso a paso, sin los ayudantes del cuaderno** (`tabla`, `contraste`, `cargar_telco`), desde el CSV crudo hasta la decisión por costo. Cada paso explica **por qué**. Debe ejecutar de principio a fin y llegar al **mismo AUC y umbral** de la Sección 4.  —  [↑ índice](#indice)

### Objetivo
Abrir la "caja negra": rehacer **manualmente el andamiaje** —cargar → limpiar → codificar → partir → ajustar → predecir → matriz de confusión → umbral por costo → decidir— para entender qué hace cada pieza y comprobar que se reproduce el resultado de la Sección 4.

### Pasos del procedimiento
1. Cargar el CSV crudo de Telco (sin `cargar_telco`).
2. Limpiar: `TotalCharges` a numérico, quitar `NaN` y `customerID`, `Churn` → 1/0.
3. Codificar one-hot y separar numéricas/categóricas.
4. Partir train/test estratificado 80/20 y escalar numéricas.
5. Ajustar la logística, predecir probabilidades y calcular el AUC (verificar que coincide).
6. Construir la matriz de confusión **manual** al 0,5.
7. Barrer umbral y **costo**; elegir el que minimiza el costo (verificar que coincide).
8. Decidir y proponer la acción de retención.

### Resultado esperado
El pipeline "desde cero" reproduce el AUC (~0,84) y el umbral óptimo (~0,1) de la Sección 4: el resultado no dependía de los ayudantes, sino del **procedimiento**.


**Paso 1 — Cargar el CSV crudo (por qué).** Se lee el archivo directamente con `pandas`, sin el ayudante `cargar_telco()`, para ver que "cargar" es solo leer un CSV. Se usan las mismas rutas/URL definidas en la preparación (`DATA`, `URL_TELCO`), que son constantes del entorno, no ayudantes de análisis.


In [ ]:
# Paso 1 — cargar el CSV crudo de Telco (SIN el ayudante cargar_telco)
import pandas as pd
ruta_telco = DATA / "telco_churn.csv"
df = pd.read_csv(ruta_telco) if ruta_telco.exists() else pd.read_csv(URL_TELCO)
print("crudo:", df.shape, "filas x columnas")
df.head(3)


**Paso 2 — Limpiar (por qué).** `TotalCharges` viene como **texto** con ~11 celdas en blanco (clientes con `tenure = 0`): se convierte a numérico (`errors="coerce"` vuelve `NaN` lo que no se puede convertir) y se eliminan esas filas. `customerID` es un identificador, no un predictor: se descarta. `Churn` (Yes/No) se vuelve 1/0 para poder modelarlo.


In [ ]:
# Paso 2 — limpiar: TotalCharges a numérico, quitar NaN y customerID, Churn -> 1/0
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
n_nan = int(df["TotalCharges"].isna().sum())
df = df.dropna(subset=["TotalCharges"]).drop(columns=["customerID"])
df["Churn"] = (df["Churn"] == "Yes").astype(int)
print(f"limpio: {df.shape[0]} clientes | {n_nan} filas con TotalCharges en blanco eliminadas "
      f"| prevalencia de fuga = {df['Churn'].mean()*100:.1f}%")


**Paso 3 — Codificar (por qué).** El modelo solo entiende números. Las **categóricas** (contrato, método de pago…) se pasan a columnas 0/1 con **one-hot** (`get_dummies`); `drop_first=True` elimina una categoría por variable para evitar la **colinealidad perfecta** entre dummies (*dummy variable trap*). Las **numéricas** se dejan como están (se escalarán en el Paso 4).


In [ ]:
# Paso 3 — one-hot de las categóricas; separar numéricas y categóricas
p_num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
p_cat = [c for c in df.columns if c not in p_num + ["Churn"]]
X = pd.get_dummies(df[p_num + p_cat], columns=p_cat, drop_first=True)
y = df["Churn"]
print("matriz de diseño:", X.shape[1], "columnas tras one-hot")


**Paso 4 — Partir y escalar (por qué).** Se separa **80 % train / 20 % test** con `stratify=y` para conservar la prevalencia de fuga en ambos lados, y `random_state=42` para que el corte sea **reproducible**. Las numéricas se **estandarizan** (media 0, desviación 1) ajustando el `StandardScaler` **solo con train** (evita fuga de información) y aplicándolo a los dos; la logística de sklearn regulariza por defecto y conviene que las escalas sean comparables.


In [ ]:
# Paso 4 — split estratificado 80/20 y escalado de numéricas (ajustado SOLO con train)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
esc = StandardScaler().fit(X_tr[p_num])
X_tr = X_tr.copy(); X_te = X_te.copy()
X_tr[p_num] = esc.transform(X_tr[p_num]); X_te[p_num] = esc.transform(X_te[p_num])
print("train:", X_tr.shape[0], " test:", X_te.shape[0])


**Paso 5 — Ajustar y evaluar (por qué).** Se entrena la **regresión logística** (`max_iter=1000` asegura la convergencia) y se predicen **probabilidades** de fuga (`predict_proba[:,1]` = P(fuga)). El **AUC** mide el poder de ordenamiento en test. Como el procedimiento es idéntico al de la Sección 4, el AUC debe **coincidir** con `auc_churn`: se comprueba con un `assert`.


In [ ]:
# Paso 5 — ajustar la logística, predecir probabilidades y AUC (debe coincidir con la Sección 4)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
mi_clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
mi_prob = mi_clf.predict_proba(X_te)[:, 1]
mi_auc = float(roc_auc_score(y_te, mi_prob))
print(f"AUC desde cero = {mi_auc:.4f}  (Sección 4: {auc_churn:.4f})")
assert np.isclose(mi_auc, auc_churn, atol=1e-6)
print("✔ Mismo AUC: el resultado no dependía de los ayudantes del cuaderno.")


**Paso 6 — Matriz de confusión manual (por qué).** En vez de llamar a `confusion_matrix`, se **cuentan** los cuatro casos con comparaciones booleanas al umbral 0,5. Así queda claro que la matriz es solo un conteo de aciertos y errores de cada tipo.


In [ ]:
# Paso 6 — matriz de confusión a mano al umbral 0,5 (contando con comparaciones booleanas)
yr = y_te.to_numpy()
yp = (mi_prob >= 0.5).astype(int)
VP = int(((yp == 1) & (yr == 1)).sum()); VN = int(((yp == 0) & (yr == 0)).sum())
FP = int(((yp == 1) & (yr == 0)).sum()); FN = int(((yp == 0) & (yr == 1)).sum())
print(f"al 0,5 -> VP={VP} FP={FP} FN={FN} VN={VN} | recall={VP/(VP+FN):.3f} precisión={VP/(VP+FP):.3f}")


**Paso 7 — Umbral por costo (por qué).** El 0,5 rara vez es óptimo. Como un **falso negativo** (perder un cliente, ~S/300) cuesta mucho más que un **falso positivo** (una llamada, ~S/20), se barre el umbral, se cuenta manualmente cada matriz y se calcula el **costo esperado** `C_FN·FN + C_FP·FP`. Se elige el umbral que **minimiza el costo** (no la exactitud). Debe coincidir con el `t*` de la Sección 4.


In [ ]:
# Paso 7 — barrer umbral y costo; elegir el que MINIMIZA el costo (debe coincidir con la Sección 4)
c_fn, c_fp = 300, 20
mejor_t, mejor_costo = None, float("inf")
print("umbral  VP  FP  FN   VN   recall   costo")
for t in np.arange(0.1, 0.91, 0.1):
    yp_t = (mi_prob >= t).astype(int)
    vp = int(((yp_t == 1) & (yr == 1)).sum()); vn = int(((yp_t == 0) & (yr == 0)).sum())
    fp = int(((yp_t == 1) & (yr == 0)).sum()); fn = int(((yp_t == 0) & (yr == 1)).sum())
    costo = c_fn * fn + c_fp * fp
    print(f"{t:5.1f}  {vp:3d} {fp:3d} {fn:3d} {vn:4d}   {vp/(vp+fn):.3f}  {costo:6d}")
    if costo < mejor_costo:
        mejor_costo, mejor_t = costo, round(float(t), 2)
print(f"\nUmbral óptimo (mínimo costo) = {mejor_t}  (Sección 4: {t_optimo})")
assert np.isclose(mejor_t, t_optimo)
print("✔ Mismo umbral óptimo, reconstruido paso a paso desde el CSV crudo.")


**Paso 8 — Decidir.** El pipeline construido desde cero llega al **mismo AUC (~0,84)** y al **mismo umbral óptimo (~0,1)** que la Sección 4: la recomendación es **bajar el umbral de 0,5 a ~0,1** para captar casi todas las fugas (recall alto) aceptando más contactos, porque el costo de perder un cliente domina. El **método** —no el atajo— es lo que produce el resultado.

> ✍️ **Tarea guiada.** Repetir el pipeline con **un** ingrediente distinto y comentar el efecto:
> 1. Usar `c_fn = 500` y `c_fp = 25` (como el drill 2): recalcular el costo por umbral y reportar el nuevo `t*` y su recall.
> 2. Cambiar `test_size=0.2` por `0.3` (mismo `random_state`): ¿cambia de forma apreciable el AUC?, ¿y el umbral óptimo?
> 3. Añadir `class_weight="balanced"` al `LogisticRegression` y comparar la matriz de confusión al 0,5 con la de arriba. (El tratamiento del desbalance por *remuestreo/pesos* es el tema de **S10**; aquí solo se observa el efecto.)


<a id="sec-9"></a>

---

## 9.9 — ¿Qué no se puede afirmar, y qué sigue en S10? Drills, entregable y conexión con S10 (Sección 9 del cuaderno)

> 🎓 **Sección 9 de 9 — En esta sección:** los tres drills, el entregable evaluable de churn y el puente hacia S10 (desbalance). — [↑ índice](#indice)


### Drills (enunciados completos en `evaluacion/drills.docx`)
1. **De coeficiente a odds ratio.** Tomar el coeficiente de `ldl` del `Logit` de SAheart, calcular su OR y su OR **por 2 unidades**, e interpretarlo (indicar si el IC del OR cruza 1).
2. **Umbral por costo de FN.** Con el churn, suponer `C_FN = S/500` y `C_FP = S/25`; calcular `t* = C_FP/(C_FP+C_FN)`, rehacer la tabla de umbral y elegir el que minimiza el costo; comparar recall y contactos frente al 0,5.
3. **GLM Poisson para reclamos.** Ajustar un Poisson (con `offset` si aplica), reportar los **rate ratios**, interpretarlos como cociente de tasas y **diagnosticar la sobre-dispersión** (`deviance/gl`).

### Entregable evaluable
Ajustar un modelo de **regresión logística de churn** (Telco): interpretar los coeficientes en **odds ratio**, evaluar con **matriz de confusión / ROC / AUC**, **elegir el umbral según el costo de los falsos negativos** y comunicar una **recomendación de retención**. Plantillas `plantillas/evaluacion_clasificador.docx` y `plantillas/guia_odds_ratio.docx`; rúbrica vigesimal (0–20) en `evaluacion/entregable.docx`.

### Para seguir explorando (las fuentes de actualidad de la sesión)
- **Telcos e IA para predecir la fuga** — *RCR Wireless News*, 16/12/2025 (la logística como línea base ≈ 88–89 % de exactitud).
- **Analítica de retención en telecom** — *Frontiers in AI*, 29/08/2025 (mismo dataset IBM: AUC 0,88, benchmark del target).
- **Churn con IA explicable (SHAP) y umbral por costo** — *Frontiers in AI*, 10/02/2026.
- **La logística, estándar del credit scoring por su explicabilidad** — *RiskSeal*, 05/08/2025 (EU AI Act, modelos *white box*).

### Conexión con la siguiente sesión
Esta sesión introduce **matriz de confusión, ROC y AUC** y deja el umbral movido **por costo**. **S10** retoma exactamente ese punto para el **desbalance de clases** (remuestreo/SMOTE, F1/PR-AUC como foco, LDA/QDA/Naive Bayes): donde hoy la palanca fue el umbral, S10 trae el **remuestreo**. Alimenta la fase de **Modelado** del proyecto integrador y el banco de ítems `[S09]`.

> **Próximas sesiones (solo se nombran).** S10 clasificación avanzada y desbalance; S11 series de tiempo; S12 causalidad; S13 supervivencia; S14 recomendación.

<a id="supuestos"></a>

---

## 9.6 (continúa) — ¿Cuándo se puede confiar? Supuestos: cómo identificarlos y corregirlos

> 🔬 **Consolidación de supuestos (logística y Poisson) — En esta sección:** para cada supuesto se dice **qué es**, **cómo identificarlo** (diagnóstico y umbral práctico) y **cómo corregirlo** (con qué método), marcando lo que es **alcance de S09** frente a lo **avanzado**. — [↑ índice](#indice)

**Fuente canónica.** El desarrollo completo de los supuestos de esta sesión vive en **la guía de supuestos de la sesión** (Parte 1 logística, Parte 2 Poisson, Parte 3 tablas-resumen). Esta sección lo **consolida** en el cuaderno y **no lo duplica**: es la contraparte operativa de los avisos ⚠️ intercalados (celdas de Poisson y de churn) y del **[Anexo A4](#anexos)**. Los diagnósticos ejecutables de más abajo se calculan **sobre los modelos ya ajustados** (`logit` de SAheart, `pois_ships`) y **no** escriben en `S09_resultados.xlsx`.

**Regla de alcance de S09.** El trabajo sobre cada supuesto llega hasta **(a) diagnosticarlo** —con su señal y un umbral práctico— y **(b) nombrar la corrección** —el método concreto en una o dos frases—. El ajuste fino de las correcciones avanzadas (Firth, modelos mixtos, GEE, binomial negativa, inflado de ceros) se nombra y se referencia, no se ejecuta en esta sesión.

### Regresión logística (GLM binomial, enlace logit)

| Supuesto | Qué es | Cómo identificarlo | Cómo corregirlo — con qué método | Alcance |
|---|---|---|---|---|
| **Linealidad en el logit** | El log-odds es lineal en cada predictor continuo (no en la probabilidad). | **Box-Tidwell** (término `x·ln x` con `p < 0,05`); residuos parciales; logit empírico por deciles. | Transformar (log/raíz/recíproco); términos polinómicos o de interacción; categorizar en tramos. *Splines / polinomios fraccionarios → avanzado.* | **S09** (diagnóstico + corrección simple) |
| **Independencia** | Cada observación aporta información no correlacionada con las demás. | Revisar el **diseño muestral** (medidas repetidas, clúster, jerarquía); correlación de residuos intra-grupo. | ES robustos por clúster; **GEE**; modelos mixtos (**GLMM**). | Diagnóstico **S09**; corrección **avanzada** |
| **Sin separación perfecta / cuasi** | Ninguna combinación de predictores clasifica sin error; si la hay, la MLE diverge (β → ±∞). | «did not converge»; β y ES desmesurados; OR extremos; probabilidades próximas a 0/1. | Quitar/combinar el predictor responsable o colapsar categorías raras; **L2 (Ridge, S05)**; **Firth** (avanzado). | Diagnóstico + quitar/combinar **S09** |
| **Sin multicolinealidad severa** | Predictores muy correlacionados entre sí: inflan los ES y vuelven inestables los OR. | **VIF** (>5 preocupa, >10 severa); correlaciones >0,8–0,9; coeficientes inestables al reajustar. | Quitar/combinar; **PCA** (bloque del curso); **L1/L2 (S05)**. | **S09** (diagnóstico + nombrar) |
| **Eventos por variable (EPV ≈ 10)** | La MLE se estabiliza con los eventos de la clase minoritaria, no con el total de filas (~10 por predictor). | `EPV = eventos(clase minoritaria) / nº predictores`; **<10 riesgo, <5 frágil** (Peduzzi 1996). | Reducir predictores; regularización (S05); conseguir más eventos. *(Desbalance en el entrenamiento → S10.)* | **S09** (control de planificación) |
| **Sin observaciones influyentes** | Casos que por sí solos cambian los coeficientes (residuo grande + leverage alto). | **Distancia de Cook** (>1, o >4/n sensible); **leverage** `hᵢ > 2p/n`; residuos estandarizados (>2, >3). | Investigar/corregir el caso (¿error de datos?); **análisis de sensibilidad** (con y sin). *Robustos → avanzado.* | Diagnóstico + sensibilidad **S09** |
| **Contraste con OLS: NO normalidad, NO homocedasticidad, NO linealidad en p** | La varianza Bernoulli `p(1−p)` la maneja la MLE (IRLS); la relación con `p` es la sigmoide, que se satura. | Es una aclaración conceptual, no un test: por eso el OLS no sirve para una respuesta binaria. | No requiere corrección: **es la razón** de usar logística y no OLS. | **S09 — central** |

### GLM Poisson (enlace log)

| Supuesto | Qué es | Cómo identificarlo | Cómo corregirlo — con qué método | Alcance |
|---|---|---|---|---|
| **Equidispersión (Var = μ)** | Poisson impone varianza = media; lo habitual es la **sobre-dispersión** (Var > μ). | **`φ̂ = deviance/gl`** (>1,5 alerta, >2 clara); dispersión de Pearson `χ²/gl`; test de Cameron-Trivedi. | **Quasi-Poisson** (escala los ES por √φ̂); **binomial negativa** (dispersión extra); efectos aleatorios por observación. | Diagnóstico + nombrar **S09**; ajuste **avanzado** |
| **Forma funcional del enlace log** | `ln(μ)` es lineal en los predictores: el efecto es multiplicativo sobre la tasa. | Residuos (deviance/Pearson) vs ajustados o vs cada predictor; *linktest* (`η̂²` significativo). | Transformar predictores / términos polinómicos. *Splines → avanzado.* | Diagnóstico + corrección simple **S09** |
| **Independencia de los conteos** | La verosimilitud supone conteos no correlacionados. | Revisar diseño (conteos agrupados/repetidos por unidad); correlación de residuos intra-grupo. | ES robustos por clúster; **GEE**; Poisson mixto (**GLMM**). *(Conteos en el tiempo → S11.)* | Diagnóstico **S09**; corrección **avanzada** |
| **Offset / exposición correcta** | Con exposición desigual se modela la **tasa**, no el conteo crudo. | Requisito de especificación: ¿las filas tienen distinto tiempo/tamaño en riesgo? | Incluir **`offset = log(exposición)`**; elegir la medida de exposición correcta. *(Aplicado con `ships`, `offset = log(service)`.)* | **S09** (se aplica) |
| **Sin exceso de ceros** | Dos mecanismos generan más ceros de los que un Poisson explica (ceros «estructurales»). | Ceros observados vs esperados (`e^(−μ̄)`); rootograma; test de Vuong. | **ZIP/ZINB** (inflado de ceros); modelos **hurdle** (dos partes). | Detectar + nombrar **S09**; modelos **avanzados** |

🔎 **Qué hace este código.** Calcula, **sin tocar el Excel**, tres diagnósticos de la logística sobre el modelo ya ajustado (`logit`, SAheart, 7 predictores): (1) el **VIF** de cada predictor (`statsmodels.variance_inflation_factor`) para la multicolinealidad; (2) el **EPV** (eventos de la clase minoritaria ÷ nº de predictores) para el tamaño muestral efectivo; y (3) la **distancia de Cook** y el **leverage** de cada caso —reajustando el mismo modelo como `GLM` binomial, que da idénticos coeficientes— para localizar observaciones influyentes. Son cálculos **de lectura**, no de resultado: no modifican `S09_resultados.xlsx`.

In [ ]:
# Diagnósticos de supuestos de la logística (SAheart) — NO escriben en el Excel
from statsmodels.stats.outliers_influence import variance_inflation_factor

# (1) Multicolinealidad: VIF de los 7 predictores (con constante para el cálculo correcto)
Xc = sm.add_constant(sa[predictores])
vif = pd.DataFrame({
    "predictor": predictores,
    "VIF": [round(variance_inflation_factor(Xc.values, i + 1), 2) for i in range(len(predictores))],
})
vif["señal"] = np.where(vif["VIF"] > 10, "severa (>10)",
                        np.where(vif["VIF"] > 5, "revisar (>5)", "ok (<5)"))
tabla(vif, "Multicolinealidad — VIF de los 7 predictores de SAheart (>5 preocupa, >10 severa)")

# (2) Eventos por variable (EPV ≈ 10): eventos de la clase minoritaria / nº de predictores
eventos = min(int(sa["chd"].sum()), int((sa["chd"] == 0).sum()))
epv = eventos / len(predictores)
estado = "holgado" if epv >= 10 else ("riesgo" if epv >= 5 else "frágil")
print(f"EPV = {eventos} eventos (clase minoritaria) / {len(predictores)} predictores = {epv:.1f}"
      f"  ->  {estado} (regla practica ~10)")

# (3) Observaciones influyentes: Cook y leverage (mismo modelo, ajustado como GLM binomial)
glm_bin = sm.GLM(sa["chd"], Xc, family=sm.families.Binomial()).fit()
infl = glm_bin.get_influence()
cook = np.asarray(infl.cooks_distance[0]); lev = np.asarray(infl.hat_matrix_diag)
n, p = len(sa), Xc.shape[1]
print(f"Cook: umbral sensible 4/n = {4/n:.4f}  ->  {int((cook > 4/n).sum())} casos a revisar "
      f"(max = {cook.max():.3f}; ninguno > 1)")
print(f"Leverage: umbral 2p/n = {2*p/n:.4f}  ->  {int((lev > 2*p/n).sum())} casos de alto leverage")

📖 **Cómo leer esta salida.**
- **VIF:** los siete valores quedan **por debajo de 5** (el mayor, `age`, ≈ 1,6): **no hay multicolinealidad** que desestabilice los odds ratio, así que cada coeficiente se interpreta por separado. Si alguno superara 5–10, se quitaría/combinaría el predictor o se resumiría el bloque con **PCA** o **L1/L2 (S05)**.
- **EPV ≈ 23:** con ~160 eventos y 7 predictores hay **holgura** frente a la regla de ~10 por variable; la MLE es estable y el riesgo de separación (β → ±∞) es bajo. Con EPV < 10 habría que reducir predictores o regularizar.
- **Cook / leverage:** **ningún** caso supera el umbral clásico de Cook > 1 (el máximo ronda 0,04); unas decenas exceden los umbrales *sensibles* (`4/n`, `2p/n`) y son candidatos a **revisar**, no a borrar. La corrección de alcance S09 es investigar si son errores de datos y hacer un **análisis de sensibilidad** (reajustar con y sin ellos y ver si cambian las conclusiones). Desarrollo completo en la guía de supuestos de la sesión (Parte 1, apartados 1.4 a 1.6).

🔎 **Qué hace este código.** Reúne el diagnóstico del supuesto **crítico** del Poisson —la **equidispersión**— sobre el GLM ya ajustado de `ships` (`pois_ships`): calcula el factor de dispersión por **deviance** (`φ̂ = deviance/gl`, ya reportado en la Sección 3) y por **Pearson** (`χ²/gl`), y los muestra con su lectura. No reajusta nada ni toca el Excel.

In [ ]:
# Diagnóstico de equidispersión del GLM Poisson (ships) — NO escribe en el Excel
gl = int(pois_ships.df_resid)
phi_dev = pois_ships.deviance / gl
phi_pear = pois_ships.pearson_chi2 / gl

def _lectura(phi):
    if phi <= 1.5:
        return "≈1 equidispersión"
    return "sobre-dispersión leve" if phi <= 2 else "sobre-dispersión clara"

disp = pd.DataFrame({
    "factor de dispersión": ["deviance / gl", "Pearson chi2 / gl"],
    "valor": [round(phi_dev, 3), round(phi_pear, 3)],
    "lectura (>1,5 alerta; >2 clara)": [_lectura(phi_dev), _lectura(phi_pear)],
})
tabla(disp, f"Equidispersión de ships — factor de dispersión por deviance y por Pearson (gl = {gl})")

📖 **Cómo leer esta salida.** Ambos factores caen en la banda de **sobre-dispersión leve** (`φ̂ ≈ 1,55` por deviance y `≈ 1,69` por Pearson: por encima de 1 pero por debajo de 2). Consecuencia práctica: los **errores estándar del Poisson simple quedan algo subestimados**, así que los p-valores se leen con prudencia. La corrección **nombrada** (no se ajusta en S09) es **quasi-Poisson** —que escala los ES por √φ̂ sin cambiar los coeficientes— o **binomial negativa**. Un caso severo de contraste es `warpbreaks` (φ ≈ 4,2). Desarrollo completo en la guía de supuestos de la sesión (Parte 2, apartado 2.1) y en el **[Anexo A4](#anexos)**.

<a id="anexos"></a>

---

## Transversal — Anexos: desarrollo matemático

> Rigor formal separado del cuerpo, al nivel de pregrado de negocios. Se colocan al final para no recargar la narrativa, pero existen para quien quiera la derivación. Notación: $p = P(Y=1\mid x)$ es la probabilidad del evento; $\eta = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k$ es el **predictor lineal**; $\mu = E[Y]$ es la media de la respuesta. Base documental: el glosario de la sesión. — [↑ índice](#indice)

- **Anexo A1.** Del odds al logit y a la función logística.
- **Anexo A2.** Verosimilitud y estimación por máxima verosimilitud (MLE).
- **Anexo A3.** El marco GLM: componente aleatorio, predictor lineal y función de enlace.
- **Anexo A4.** Supuestos de cada modelo y su diagnóstico.
- **Anexo A5.** Interpretación de coeficientes: $\exp(\beta)$ como odds ratio y rate ratio.

### Anexo A1 — Del odds al logit y a la función logística

Sea $p = P(Y=1\mid x)$ la probabilidad del evento. Se definen los **odds** (razón de momios) y el **logit** (su logaritmo):

$$\operatorname{odds}(p) = \frac{p}{1-p} \in [0, \infty), \qquad \operatorname{logit}(p) = \ln\!\left(\frac{p}{1-p}\right) \in (-\infty, +\infty).$$

El logit lleva el intervalo acotado $(0,1)$ a **toda** la recta real, lo que permite igualarlo a un predictor lineal sin restricciones de rango:

$$\operatorname{logit}(p) = \eta = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k.$$

Despejando $p$ se obtiene la inversa del logit, la **función logística** o **sigmoide**:

$$p = \sigma(\eta) = \frac{1}{1 + e^{-\eta}} = \frac{e^{\eta}}{1 + e^{\eta}}.$$

Como $0 < \sigma(\eta) < 1$ para todo $\eta \in \mathbb{R}$, la predicción **siempre** es una probabilidad válida (a diferencia de la recta del OLS). Su derivada, útil en el Anexo A2, es $\sigma'(\eta) = \sigma(\eta)\,\big[1 - \sigma(\eta)\big]$.

### Anexo A2 — Verosimilitud y estimación por máxima verosimilitud (MLE)

Con $n$ observaciones independientes $(x_i, y_i)$, $y_i \in \{0,1\}$, cada $Y_i$ es Bernoulli con $p_i = \sigma(\eta_i)$ y $\eta_i = x_i^{\top}\beta$. La **verosimilitud** (probabilidad conjunta de los datos bajo el modelo) es:

$$L(\beta) = \prod_{i=1}^{n} p_i^{\,y_i}\,(1 - p_i)^{\,1 - y_i}.$$

Tomando logaritmo se obtiene la **log-verosimilitud**, más cómoda de maximizar:

$$\ell(\beta) = \sum_{i=1}^{n} \Big[\, y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \,\Big].$$

Los estimadores $\hat\beta$ **maximizan** $\ell(\beta)$. Derivando e igualando a cero se llega a las **ecuaciones de verosimilitud**:

$$\frac{\partial \ell}{\partial \beta} = \sum_{i=1}^{n} \big(y_i - p_i\big)\,x_i = X^{\top}(y - p) = \mathbf{0},$$

que **no** tienen solución cerrada, porque $p_i$ depende de $\beta$ de forma no lineal. Se resuelven numéricamente por **Newton–Raphson** o, equivalentemente, por **mínimos cuadrados reponderados iterativamente (IRLS)** — el mismo algoritmo que unifica a todos los GLM. A diferencia del OLS, que **minimiza** $\sum_i (y_i - \hat y_i)^2$, aquí se **maximiza** una verosimilitud; de la curvatura de $\ell$ (la inversa de la matriz de información) se obtienen además los **errores estándar** y, con ellos, los estadísticos $z$, los p-valores y los IC. Dos resúmenes de ajuste derivan de $\ell$: la **deviance** $D = -2\big[\ell(\hat\beta) - \ell_{\text{sat}}\big]$ y el **pseudo-R² de McFadden** $R^2_{\text{McF}} = 1 - \ell(\hat\beta)/\ell_0$, con $\ell_0$ la del modelo solo-intercepto.

### Anexo A3 — El marco GLM: componente aleatorio, predictor lineal y función de enlace

Un **Modelo Lineal Generalizado (GLM)** se arma con tres piezas:

1. **Componente aleatorio.** $Y$ sigue una distribución de la **familia exponencial** con media $\mu = E[Y]$ (Normal, Binomial, Poisson, Gamma, …).
2. **Componente sistemático (predictor lineal).** $\eta = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k = x^{\top}\beta$.
3. **Función de enlace** $g$, monótona e invertible, que conecta la media con el predictor lineal:

$$g(\mu) = \eta \qquad \Longleftrightarrow \qquad \mu = g^{-1}(\eta).$$

Cada familia tiene un **enlace canónico**. Los dos casos de esta sesión:

| Caso | Distribución | Enlace canónico $g(\mu)$ | Media $\mu = g^{-1}(\eta)$ |
|---|---|---|---|
| **Logística** (binaria) | Binomial | $\operatorname{logit}(\mu) = \ln\dfrac{\mu}{1-\mu}$ | $\mu = \dfrac{1}{1 + e^{-\eta}}$ |
| **Poisson** (conteos) | Poisson | $\ln(\mu)$ | $\mu = e^{\eta}$ |

La **regresión lineal** es el caso particular con familia Normal y enlace **identidad** ($g(\mu)=\mu$). Con **offset** (exposición desigual) el Poisson modela una **tasa**: $\ln(\mu) = \ln(t) + x^{\top}\beta$, es decir $\mu = t\,e^{x^{\top}\beta}$, donde $t$ es la exposición (p. ej. meses de servicio del casco). Todos los GLM se estiman por MLE con IRLS (Anexo A2).

### Anexo A4 — Supuestos de cada modelo y su diagnóstico

> **Desarrollo completo:** la fuente canónica de los supuestos de S09 es la guía de supuestos de la sesión (Parte 1 logística, Parte 2 Poisson); su consolidación operativa en el cuaderno es la sección **[Supuestos — cómo identificarlos y corregirlos](#supuestos)**.

**Regresión logística.**
- *Independencia* de las observaciones.
- *Linealidad en el logit*: $\operatorname{logit}(p)$ es lineal en los predictores (no en $p$). Se diagnostica con residuos frente al predictor o con términos suavizados; se corrige con transformaciones o términos polinómicos.
- *Sin multicolinealidad severa* (revisar el VIF) y *sin separación perfecta*: si una variable separa perfectamente las clases, la MLE diverge (coeficientes $\to \pm\infty$).
- *Suficientes eventos por variable (EPV ≈ 10)*: la estabilidad de la MLE depende del número de **eventos** de la clase minoritaria, no del total de filas (~10 por predictor, Peduzzi et al. 1996). Se diagnostica con `EPV = eventos / nº de predictores` (<10 riesgo, <5 frágil); se corrige reduciendo predictores, con regularización L1/L2 (S05) o consiguiendo más eventos. **[Alcance S09]**
- *Sin observaciones influyentes / atípicas*: casos que por sí solos cambian los coeficientes (residuo grande junto a leverage alto). Se diagnostica con la **distancia de Cook** ($>1$, o $>4/n$ como regla sensible), el **leverage** ($h_i > 2p/n$) y los residuos estandarizados (>2 revisar, >3 atípico); en statsmodels, `GLM(...).get_influence`. Se corrige investigando el caso (¿error de datos?) y con un **análisis de sensibilidad** (reajustar con y sin él). **[Alcance S09]**
- **No** se exigen normalidad ni homocedasticidad de los residuos (eso era del OLS): la varianza es la del Bernoulli, $p(1-p)$, y así se estima por MLE.

**GLM Poisson.**
- *Media igual a la varianza*: $\operatorname{Var}(Y) = \mu$. Es el supuesto crítico.
- *Enlace log correcto* e *independencia* de los conteos.
- **Sobre-dispersión** (el fallo más común): si $\operatorname{Var}(Y) > \mu$, se detecta con $\hat\varphi = D / \text{gl}$ (deviance sobre grados de libertad); regla práctica $\hat\varphi > 1{,}5$. Se corrige con **quasi-Poisson** (escala los errores estándar por $\sqrt{\hat\varphi}$) o con **binomial negativa** (se nombra; el ajuste fino es de modelado avanzado).
- *Sin exceso de ceros* (zero-inflation): algunos procesos generan más ceros de los que un Poisson explica (ceros «estructurales»). Se detecta comparando los ceros observados con los esperados ($e^{-\bar\mu}$); la corrección se **nombra** —modelos inflados de ceros (ZIP/ZINB) o *hurdle* de dos partes— y queda como material avanzado, fuera del alcance de S09.

**Por qué el OLS no aplica a estos casos.** Sobre una respuesta binaria o de conteo, el OLS viola: (i) el **rango** de la predicción (excede $[0,1]$ o predice conteos negativos), (ii) la **normalidad** de los errores, y (iii) la **homocedasticidad** —la varianza real depende de la media: $p(1-p)$ en Bernoulli, $\mu$ en Poisson—, además de imponer un efecto **aditivo constante** donde el fenómeno es multiplicativo y se satura.

### Anexo A5 — Interpretación de coeficientes: $\exp(\beta)$ como odds ratio y rate ratio

**Logística (odds ratio).** Al subir $x_j$ en una unidad, con lo demás fijo, el logit cambia en $\beta_j$, de modo que los **odds** se multiplican por $e^{\beta_j}$:

$$\frac{\operatorname{odds}(x_j + 1)}{\operatorname{odds}(x_j)} = \frac{e^{\beta_0 + \dots + \beta_j (x_j + 1) + \dots}}{e^{\beta_0 + \dots + \beta_j x_j + \dots}} = e^{\beta_j} \equiv \text{OR}.$$

Lectura: $\text{OR} > 1$ = factor de riesgo; $\text{OR} < 1$ = protector; $\text{OR} = 1$ (es decir $\beta_j = 0$) = sin efecto. Para un cambio de $d$ unidades, el odds ratio es $e^{d\beta_j}$. Advertencias: un OR **no** es un incremento de probabilidad (la sigmoide es no lineal, el efecto en $p$ depende del nivel base) ni un efecto **causal** (ver Sección 6, "Asociación ≠ causa"; la causalidad formal es S12).

**Poisson (rate ratio).** Con enlace log, $\mu = e^{x^{\top}\beta}$; al subir $x_j$ en una unidad la **tasa** esperada se multiplica por $e^{\beta_j}$, el **rate ratio**:

$$\frac{\mu(x_j + 1)}{\mu(x_j)} = e^{\beta_j} \equiv \text{RR}.$$

Es la misma lectura multiplicativa del OR, sobre otra familia: *"cada unidad de $x_j$ multiplica por $e^{\beta_j}$ la tasa de eventos"*. El **intercepto** exponenciado, $e^{\beta_0}$, es la tasa (o los odds) de la **categoría de referencia**.